# 17 — Strength and speed benchmarks

Open this notebook and click **Run All** to launch the benchmark in detached tmux. The current teacher-trained AHR champion stays frozen; this notebook performs no training.

**Inputs:** selected step-8016 playing checkpoint from notebook 15, the existing prepared teacher validation source for timing positions, notebooks 00–03/05/08/12–16 tagged definitions, GNU Go 3.8 and pinned official KataGo CPU assets.

**Outputs:** reports in `reports/ahr_v2/benchmarks/<run_id>/`; full resumable checkpoints in `checkpoints/<run_id>/`, `<run_id>-plan/` and `<run_id>-speed/`.

**Runtime:** allow several hours for 100 games; a reliable ETA needs measured game lengths. Each game has a 15-minute play cutoff and a 600-ply cutoff, recorded as UNKNOWN. Setup, replay and checkpoint time add overhead. Timing probes take a few minutes.

Saved outputs below describe short validation only. The full benchmark has not been launched during notebook preparation.

## Setup and settings

Use the **AHR Go** kernel. The definition cells load only tagged exports and never execute upstream training cells.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'AGENTS.md').is_file() and PROJECT!=PROJECT.parent:
    PROJECT=PROJECT.parent
if not (PROJECT/'experiments/ahr_v2').is_dir():
    raise RuntimeError('Open notebook 17 from the AHR Go workspace.')
for name in ['00_contracts_and_recovery.ipynb','15_teacher_evaluation_and_launch.ipynb']:
    for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,name+':ahr-export','exec'),globals())
globals().update(go2_teacher_arena_namespace(PROJECT))
import pandas as pd
from IPython.display import display, Markdown, HTML
from zoneinfo import ZoneInfo

torch.set_num_threads(4)


In [ ]:
import shlex, shutil, subprocess, uuid, datetime, html

GO2_BENCH_BOOK='17_strength_and_speed_benchmarks.ipynb'

def go2_bench_sources(project):
    sources=go2_teacher_control_sources(project)
    sources[GO2_ROOT+'/'+GO2_BENCH_BOOK]=go2_code_sha(Path(project)/GO2_ROOT/GO2_BENCH_BOOK)
    return sources

def go2_bench_default_config():
    return {'run_id':'ahr-v2-benchmark-20261009-v1','purpose':'development',
            'training_run_id':'ahr-v2-teacher-20261008','candidate_file':'step-000008016-c19b207e.pt',
            'seed':2026100917,'board_size':9,'komi':7.5,'device':'cuda:0',
            'ahr_move_seconds':2.0,'max_simulations':512,'batch_size':8,
            'max_plies':600,'game_time_seconds':900.0,'gtp_timeout_seconds':30.0,
            'torch_threads':4,'reading_total_nodes':256,'reading_quantum':16,'reading_wall_time_seconds':0.05,
            'opening_plies':8,'opening_temperature':1.25,'empty_board_baseline':True,
            'max_opening_attempts':10000,
            'profiles':[{'id':'gnugo-l10','kind':'gnugo','level':10,'games':40},
                        {'id':'katago-v1','kind':'katago','visits':1,'games':20},
                        {'id':'katago-v4','kind':'katago','visits':4,'games':20},
                        {'id':'katago-v16','kind':'katago','visits':16,'games':20}],
            'speed_devices':['cpu','cuda:0'],'speed_move_seconds':[1.0,2.0],
            'speed_positions':12,'speed_max_simulations':512}

def go2_bench_pinned_model(project,config,device=None):
    model,metadata=go2_load_selected_teacher_player(project,run_id=config['training_run_id'],
        candidate_file=config['candidate_file'],device=device or config['device'])
    if metadata['is_smoke'] or metadata['optimizer_step']<=0:
        raise RuntimeError('A real trained playing checkpoint is required.')
    return model,metadata

def go2_bench_frozen_config(project,config):
    config=copy.deepcopy(config)
    if Path(config['run_id']).name!=config['run_id'] or not config['run_id'].startswith('ahr-v2-benchmark-'):
        raise ValueError('Use a new local ahr-v2-benchmark- run ID for a different configuration.')
    if config.get('purpose')!='development' or config.get('board_size')!=9:
        raise ValueError('Notebook 17 is a 9×9 development benchmark.')
    if config['ahr_move_seconds']<=0 or config['max_simulations']<1:
        raise ValueError('Positive search time and simulation budgets are required.')
    if config['device'].startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('Configured GPU unavailable; do not silently switch hardware.')
    if not config['profiles'] or len({p['id'] for p in config['profiles']})!=len(config['profiles']):
        raise ValueError('Distinct opponent profiles are required.')
    for p in config['profiles']:
        if p['id'] not in GO2_BENCH_PROFILES:raise ValueError('Unknown opponent profile.')
        spec=GO2_BENCH_PROFILES[p['id']]
        if p['kind']!=spec['engine'] or any(p.get(k)!=spec[k] for k in ['level','visits'] if k in spec):
            raise ValueError('Profile description must match the actual pinned opponent setting.')
        if type(p['games']) is not int or p['games']<2 or p['games']%2:raise ValueError('Use even positive game counts.')
    if not config['speed_devices'] or not config['speed_move_seconds'] or config['speed_positions']<3:
        raise ValueError('At least three timing positions, a device and a budget are required.')
    if any(not isinstance(b,(int,float)) or b<=0 for b in config['speed_move_seconds']):raise ValueError('Positive timing budgets required.')
    model,metadata=go2_bench_pinned_model(project,config,'cpu')
    config['source_identity']=go2_bench_sources(project)
    config['champion_pin']={key:metadata[key] for key in ('run_id','optimizer_step','model_identity','checkpoint_file','checkpoint_sha256')}
    config['opponent_pins']={p['id']:go2_bench_opponent_pin(project,p['id']) for p in config['profiles']}
    return config

def go2_bench_dublin(utc_time):
    if not utc_time:return '—'
    parsed=datetime.datetime.fromisoformat(utc_time.replace('Z','+00:00'))
    if parsed.tzinfo is None:parsed=parsed.replace(tzinfo=datetime.timezone.utc)
    return parsed.astimezone(ZoneInfo('Europe/Dublin')).strftime('%d %b %Y %H:%M:%S %Z')


In [ ]:
# Leave these defaults for the first benchmark. Use a new RUN_ID if budgets change.
EXECUTION_MODE=os.environ.get('AHR_BENCH17_MODE','benchmark')
BENCHMARK_CONFIG=go2_bench_default_config()
TMUX_SESSION='ahr-v2-benchmark17'
RESUME_STOPPED_RUN=False
assert EXECUTION_MODE in ('benchmark','validate')
print('Execution mode:',EXECUTION_MODE)
print('Champion optimizer step: 8016; planned games:',sum(p['games'] for p in BENCHMARK_CONFIG['profiles']))


## Benchmark method

The suite pairs Black and White on each opening: 40 games versus GNU Go level 10, then 20 each versus KataGo at visit caps 1, 4 and 16. Each opponent also gets one empty-board baseline pair. Other openings are legal eight-ply samples from the frozen Scout, unique by board symmetry and excluded from known training/development opening families and earlier GNU reviews.

AHR uses a two-second **soft** search target with at most 512 simulations. End-to-end measurements include features, neural inference, tactical reading and CUDA synchronization; atomic work can overrun. CPU KataGo uses fixed visits, so these results compare configured players with different compute budgets.

The speed stage replays opening, middle and late positions from compatible raw **validation** games. It measures CPU/GPU at one- and two-second targets, including visits achieved and overruns. No test payload is read and no new training label is made.

Scores use 9×9 area, komi 7.5, positional superko and no suicide. Games end after two passes; there is no dead-stone adjudication. Bounded tactical findings do not prove a full-game win.

## Opponent assets and transport

Official assets: [KataGo v1.18.1 engine](https://github.com/lightvector/KataGo/releases/tag/v1.18.1), [dedicated 9×9 network](https://github.com/lightvector/KataGo/releases/tag/v1.13.2-kata9x9). They are installed under `.venv/katago-benchmark17/` and verified against a pinned manifest.

In [ ]:
import hashlib, json, os, shutil, subprocess, tempfile, time, urllib.request, zipfile
from pathlib import Path

GO2_BENCH_KATA_ARCHIVE_URL = 'https://github.com/lightvector/KataGo/releases/download/v1.18.1/katago-v1.18.1-eigenavx2-linux-x64.zip'
GO2_BENCH_KATA_ARCHIVE_SHA = '33e79780dbe3bf6ee859e16f64952cdfc90f7210c8f71ad978ffcba85ad20d79'
GO2_BENCH_KATA_MODEL_URL = 'https://github.com/lightvector/KataGo/releases/download/v1.13.2-kata9x9/kata9x9-b18c384nbt-20231025.bin.gz'
GO2_BENCH_KATA_MODEL_SHA = 'a1298ce1adc1dad7bd868ca962b2384cc8388ed373a00e6bae1114fa6f9e2d61'
GO2_BENCH_KATA_RUNTIME_SHA = {'v1.18.1-eigenavx2/katago': '05c3450274bb8008e57e3011a7c80eb6e370ea395d29ab49c400bda9473766b1', 'v1.18.1-eigenavx2/squashfs-root/AppRun': '2ddcaa8755c73b980e895d1466fa8589b998c47686846a995fe3a8e258b4c122', 'v1.18.1-eigenavx2/squashfs-root/usr/bin/katago': 'c0ed553a775fd9af585717ff00ddeb5007bf15d283caf148d237e0ab24f6f308', 'v1.18.1-eigenavx2/squashfs-root/usr/lib/libbz2.so.1.0': '5e516f77fc36dd924fdf02c8489a217f55fa1548883d32c3a5e041fb25d47d6e', 'v1.18.1-eigenavx2/squashfs-root/usr/lib/libz.so.1': '64c206f0146cc58bbddc4f22054436f4ff278f5a554aa3ce6921ddf7e9133370', 'v1.18.1-eigenavx2/squashfs-root/usr/lib/libzip.so.4': '05746c16c79c34f3b408c5a6a5ddf9bd18e61c1cfd35f587e215ea37f23fe4ad'}

def _go2_bench_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def _go2_bench_download(url, path, expected_sha):
    path = Path(path)
    if path.exists():
        actual = _go2_bench_sha(path)
        if expected_sha is not None and actual != expected_sha:
            raise RuntimeError('Existing KataGo dependency hash mismatch: ' + str(path))
        return actual
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.part-' + str(os.getpid()))
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'AHR-Go-notebook17/1.0'})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open('xb') as target:
            shutil.copyfileobj(response, target, length=1024 * 1024)
            target.flush(); os.fsync(target.fileno())
        actual = _go2_bench_sha(temporary)
        if expected_sha is not None and actual != expected_sha:
            raise RuntimeError('Downloaded KataGo dependency hash mismatch.')
        os.replace(temporary, path)
        return actual
    finally:
        if temporary.exists(): temporary.unlink()

def go2_bench_katago_assets(project=None, allow_download=True):
    project = Path(project or '/home/benjamin/workspace/ahr_go').resolve()
    directory = project / '.venv' / 'katago-benchmark17'
    archive = directory / 'katago-v1.18.1-eigenavx2-linux-x64.zip'
    model = directory / 'kata9x9-b18c384nbt-20231025.bin.gz'
    executable = directory / 'v1.18.1-eigenavx2' / 'katago'
    manifest_path = directory / 'manifest.json'
    if not allow_download and (not archive.is_file() or not model.is_file() or not executable.is_file() or not manifest_path.is_file()):
        raise FileNotFoundError('KataGo assets are not prepared. Run notebook 17 setup with downloads enabled.')
    archive_sha = _go2_bench_download(GO2_BENCH_KATA_ARCHIVE_URL, archive, GO2_BENCH_KATA_ARCHIVE_SHA)
    model_sha = _go2_bench_download(GO2_BENCH_KATA_MODEL_URL, model, GO2_BENCH_KATA_MODEL_SHA)
    if not executable.exists():
        executable.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            names = [name for name in bundle.namelist() if Path(name).name == 'katago']
            if len(names) != 1: raise RuntimeError('Official archive has an unexpected executable inventory.')
            data = bundle.read(names[0])
        temporary = executable.with_name('katago.part-' + str(os.getpid()))
        with temporary.open('xb') as stream:
            stream.write(data); stream.flush(); os.fsync(stream.fileno())
        temporary.chmod(0o755); os.replace(temporary, executable)
    # The official Linux archive packages an AppImage. Extract it once inside
    # the environment, avoiding FUSE and any system-wide installation.
    appimage = executable
    extracted = appimage.parent / 'squashfs-root'
    executable = extracted / 'AppRun'
    if not executable.is_file():
        subprocess.run([str(appimage), '--appimage-extract'], cwd=appimage.parent,
                       capture_output=True, check=True, timeout=60)
    dependency_files = [appimage, executable, extracted / 'usr' / 'bin' / 'katago',
                        *sorted((extracted / 'usr' / 'lib').glob('*'))]
    if any(not path.is_file() for path in dependency_files):
        raise RuntimeError('Incomplete extracted KataGo runtime.')
    dependency_hashes = {str(path.relative_to(directory)): _go2_bench_sha(path)
                         for path in dependency_files}
    if dependency_hashes != GO2_BENCH_KATA_RUNTIME_SHA:
        raise RuntimeError('KataGo runtime hash mismatch; refuse execution.')
    version_result = subprocess.run([str(executable), 'version'], capture_output=True, text=True, check=True, timeout=15)
    version = version_result.stdout.strip()
    if not version.startswith('KataGo v1.18.1'): raise RuntimeError('Unexpected installed KataGo version: ' + version)
    actual = {'schema': 'ahr-go-katago-assets-v1', 'backend': 'EIGEN-AVX2-CPU',
              'executable': str(executable), 'executable_sha256': _go2_bench_sha(executable),
              'version': version, 'archive_url': GO2_BENCH_KATA_ARCHIVE_URL,
              'archive_sha256': archive_sha, 'model': str(model), 'model_sha256': model_sha,
              'model_url': GO2_BENCH_KATA_MODEL_URL, 'model_board_size': 9,
              'model_release': 'v1.13.2-kata9x9', 'runtime_sha256': dependency_hashes}
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text())
        if previous != actual: raise RuntimeError('KataGo asset manifest mismatch; silent replacement refused.')
    else:
        temporary = manifest_path.with_name('manifest.json.part-' + str(os.getpid()))
        with temporary.open('x') as stream:
            json.dump(actual, stream, sort_keys=True, indent=2); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary, manifest_path)
    return actual


In [ ]:
import dataclasses, math, selectors, types

GO2_BENCH_KATA_RULES = {'ko': 'POSITIONAL', 'scoring': 'AREA', 'tax': 'NONE',
                        'suicide': False, 'hasButton': False,
                        'whiteHandicapBonus': '0', 'friendlyPassOk': False}
GO2_BENCH_PROFILES = {'gnugo-l10': {'engine': 'gnugo', 'level': 10},
                     'katago-v1': {'engine': 'katago', 'visits': 1},
                     'katago-v4': {'engine': 'katago', 'visits': 4},
                     'katago-v16': {'engine': 'katago', 'visits': 16}}

def go2_bench_opponent_pin(project, profile):
    if profile not in GO2_BENCH_PROFILES: raise ValueError('Unknown benchmark opponent profile.')
    spec = GO2_BENCH_PROFILES[profile]
    if spec['engine'] == 'gnugo':
        executable = Path(shutil.which('gnugo') or '/usr/games/gnugo').resolve()
        if not executable.is_file(): raise FileNotFoundError('GNU Go dependency missing.')
        return {'profile': profile, 'name': 'GNU Go', 'level': spec['level'],
                'executable': str(executable), 'sha256': _go2_bench_sha(executable),
                'version': subprocess.run([str(executable), '--version'], capture_output=True,
                                          text=True, check=True, timeout=10).stdout.splitlines()[0],
                'rules': GO2_BENCH_KATA_RULES, 'backend': 'CPU',
                'budget_type': 'GNU Go level setting; no move-time control'}
    assets = go2_bench_katago_assets(project, allow_download=False)
    return {'profile': profile, 'name': 'KataGo 9×9', 'max_visits': spec['visits'],
            'executable': assets['executable'], 'sha256': assets['executable_sha256'],
            'version': assets['version'], 'model': assets['model'],
            'model_sha256': assets['model_sha256'], 'runtime_sha256': assets['runtime_sha256'],
            'backend': assets['backend'], 'rules': GO2_BENCH_KATA_RULES,
            'budget_type': 'fresh search visit cap; single CPU search and inference thread',
            'equal_wall_time_to_ahr': False}

def _go2_bench_gtp_vertex(action, size):
    if action == size * size: return 'pass'
    if type(action) is not int or not 0 <= action < size * size: raise ValueError('Invalid action.')
    row, col = divmod(action, size)
    return 'ABCDEFGHJKLMNOPQRST'[col] + str(size - row)

def _go2_bench_gtp_action(vertex, size):
    text = vertex.strip().upper()
    if text == 'PASS': return size * size
    if text == 'RESIGN': raise Go2OpponentError('Resignation is UNKNOWN, not an exact scored game.')
    if len(text) < 2 or text[0] not in 'ABCDEFGHJKLMNOPQRST'[:size]:
        raise Go2OpponentError('Invalid opponent GTP coordinate.')
    try: row = int(text[1:])
    except ValueError as exc: raise Go2OpponentError('Invalid GTP row.') from exc
    if not 1 <= row <= size: raise Go2OpponentError('Opponent GTP coordinate outside board.')
    return (size - row) * size + 'ABCDEFGHJKLMNOPQRST'.index(text[0])

class _Go2BenchGTPTransport:
    def _start(self, argv, config, cwd):
        self.config = config; self.sequence = 0; self.buffer = bytearray()
        stderr_directory = Path(getattr(config, 'config_directory', cwd))
        stderr_directory.mkdir(parents=True, exist_ok=True)
        self.stderr_path = stderr_directory / ('opponent-' + str(os.getpid()) + '-' + str(time.monotonic_ns()) + '.log')
        self.stderr_stream = self.stderr_path.open('xb')
        self.process = subprocess.Popen(argv, cwd=cwd, stdin=subprocess.PIPE,
                                        stdout=subprocess.PIPE, stderr=self.stderr_stream, bufsize=0)
        self.selector = selectors.DefaultSelector()
        self.selector.register(self.process.stdout, selectors.EVENT_READ)
    def command(self, command):
        if not command or '\n' in command or '\r' in command: raise ValueError('Invalid GTP command.')
        self.sequence += 1; number = self.sequence
        if self.process.poll() is not None: raise Go2OpponentError('Opponent exited before responding.')
        try:
            self.process.stdin.write((str(number) + ' ' + command + '\n').encode())
            self.process.stdin.flush()
        except (BrokenPipeError, OSError) as exc:
            raise Go2OpponentError('Opponent GTP pipe closed.') from exc
        deadline = time.monotonic() + self.config.gtp_timeout_seconds
        while b'\n\n' not in self.buffer:
            remaining = deadline - time.monotonic()
            if remaining <= 0 or not self.selector.select(max(0, remaining)):
                self.close(); raise TimeoutError('Opponent response timeout: ' + command.split()[0])
            chunk = os.read(self.process.stdout.fileno(), 65536)
            if not chunk: raise Go2OpponentError('Opponent closed its output.')
            self.buffer.extend(chunk.replace(b'\r\n', b'\n'))
            if len(self.buffer) > 32 * 1024 * 1024:
                self.close(); raise Go2OpponentError('Opponent response exceeds bounded transport size.')
        raw, _, tail = self.buffer.partition(b'\n\n'); self.buffer = bytearray(tail)
        response = raw.decode('utf-8').strip(); prefix = '=' + str(number)
        if not response.startswith(prefix) or (len(response) > len(prefix) and not response[len(prefix)].isspace()):
            raise Go2OpponentError('Opponent rejected/mismatched command: ' + response[:200])
        return response[len(prefix):].strip()
    def play(self, color, action):
        if color not in (1, -1): raise ValueError('Invalid player color.')
        self.command('play ' + ('black' if color == 1 else 'white') + ' ' +
                     _go2_bench_gtp_vertex(action, self.config.board_size))
    def replay(self, trace):
        for item in trace:
            if item['kind'] == 'play': self.play(item['color'], item['action'])
            elif item['kind'] == 'genmove':
                regenerated = self.genmove(item['color'])
                if regenerated != item['action']:
                    self.close(); raise Go2OpponentError('Opponent regenerated a different move; exact resume refused.')
            else: raise RuntimeError('Unknown opponent replay command kind.')
    def snapshot(self):
        return {'pin': self.pin, 'seed': self.seed, 'board_size': self.config.board_size,
                'komi': self.config.komi, 'runtime_rules': getattr(self, 'runtime_rules', None),
                'runtime_parameters': getattr(self, 'runtime_parameters', None)}
    def close(self):
        process = getattr(self, 'process', None)
        if process is not None:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=2)
                except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=2)
            for stream in (process.stdin, process.stdout):
                if stream is not None: stream.close()
        selector = getattr(self, 'selector', None)
        if selector is not None: selector.close()
        stderr_stream = getattr(self, 'stderr_stream', None)
        if stderr_stream is not None: stderr_stream.close()
    def __enter__(self): return self
    def __exit__(self, *_): self.close()

class Go2BenchGNU(_Go2BenchGTPTransport):
    def __init__(self, config, seed, pin):
        self.pin = pin; self.seed = int(seed)
        flags = ['--mode','gtp','--level',str(pin['level']),'--seed',str(self.seed),
                 '--chinese-rules','--positional-superko','--forbid-suicide',
                 '--never-resign','--capture-all-dead']
        self._start([pin['executable'], *flags], config, config.project)
        try:
            self.version = self.command('version')
            self.command('boardsize ' + str(config.board_size)); self.command('clear_board')
            self.command('komi ' + str(config.komi))
        except BaseException: self.close(); raise
    def genmove(self, color):
        if color not in (1, -1): raise ValueError('Invalid player color.')
        start = time.monotonic()
        move = self.command('genmove ' + ('black' if color == 1 else 'white'))
        self.last_move_stats = {'wall_seconds': time.monotonic() - start, 'level': self.pin['level']}
        return _go2_bench_gtp_action(move, self.config.board_size)

class Go2BenchKataGo(_Go2BenchGTPTransport):
    def __init__(self, config, seed, assets):
        self.seed = int(seed); self.pin = assets
        if config.board_size != 9: raise ValueError('Dedicated KataGo reference network requires a 9×9 board.')
        directory = Path(config.config_directory)
        directory.mkdir(parents=True, exist_ok=True)
        spec = {'koRule':'POSITIONAL','scoringRule':'AREA','taxRule':'NONE',
                'multiStoneSuicideLegal':False,'hasButton':False,'friendlyPassOk':False,
                'whiteHandicapBonus':'0','allowResignation':False,'resignThreshold':-0.9,
                'resignConsecTurns':3,'conservativePass':False,
                'preventCleanupPhase':True,'defaultBoardSize':9,
                'assumeMultipleStartingBlackMovesAreHandicap':False,
                'maxVisits':assets['max_visits'],'numSearchThreads':1,'numEigenThreadsPerModel':1,
                'nnMaxBatchSize':1,'nnCacheSizePowerOfTwo':12,'nnMutexPoolSizePowerOfTwo':8,
                'ponderingEnabled':False,'nnRandomize':False,'searchRandSeed':str(self.seed),
                'nnRandSeed':str(self.seed),'chosenMoveTemperature':0,
                'chosenMoveTemperatureEarly':0,'rootNoiseEnabled':False,
                'rootNumSymmetriesToSample':1,'rootSymmetryPruning':False,
                'searchFactorAfterOnePass':1,'searchFactorAfterTwoPass':1,
                'searchFactorWhenWinning':1,'useEvalCache':False,
                'dynamicPlayoutDoublingAdvantageCapPerOppLead':0,
                'logToStderr':False,'startupPrintMessageToStderr':False,
                'logAllGTPCommunication':False,'logSearchInfo':False}
        text = '\n'.join(key + ' = ' + (str(value).lower() if isinstance(value,bool) else str(value))
                         for key,value in spec.items()) + '\n'
        digest = hashlib.sha256(text.encode()).hexdigest()
        self.config_path = directory / ('katago-' + digest[:20] + '.cfg')
        if self.config_path.exists() and self.config_path.read_text() != text:
            raise RuntimeError('KataGo per-game config hash collision.')
        if not self.config_path.exists():
            temporary = self.config_path.with_name(self.config_path.name + '.part-' + str(os.getpid()))
            with temporary.open('x') as stream:
                stream.write(text); stream.flush(); os.fsync(stream.fileno())
            os.replace(temporary, self.config_path)
        self._start([assets['executable'],'gtp','-model',assets['model'],'-config',str(self.config_path)],
                    config, directory)
        try:
            self.version = self.command('version')
            self.command('boardsize ' + str(config.board_size)); self.command('clear_board')
            self.command('komi ' + str(config.komi))
            self.runtime_rules = json.loads(self.command('kata-get-rules'))
            if any(self.runtime_rules.get(key) != value for key,value in GO2_BENCH_KATA_RULES.items()):
                raise Go2OpponentError('KataGo runtime rules do not match AHR.')
            if float(self.command('get_komi')) != config.komi: raise Go2OpponentError('KataGo komi mismatch.')
            self.runtime_parameters = json.loads(self.command('kata-get-params'))
            for key,expected in [('maxVisits',assets['max_visits']),('numSearchThreads',1),
                                 ('chosenMoveTemperature',0),('chosenMoveTemperatureEarly',0),
                                 ('rootNoiseEnabled',False)]:
                if self.runtime_parameters.get(key) != expected:
                    raise Go2OpponentError('KataGo runtime parameter mismatch: ' + key)
        except BaseException: self.close(); raise
    def genmove(self, color):
        if color not in (1, -1): raise ValueError('Invalid player color.')
        # Fresh root avoids reusing visits from earlier turns. The cap is the
        # queried runtime maxVisits, not an unobserved exact count of new playouts.
        self.command('clear_cache')
        start = time.monotonic()
        move = self.command('genmove ' + ('black' if color == 1 else 'white'))
        self.last_move_stats = {'wall_seconds':time.monotonic() - start,
                               'max_visits':self.runtime_parameters['maxVisits'],
                               'actual_visits':None,'fresh_search':True,'backend':self.pin['backend']}
        return _go2_bench_gtp_action(move, self.config.board_size)

def go2_bench_opponent_factory(project, profile, seed, board_size=9, komi=7.5,
                              gtp_timeout_seconds=90, config_directory=None):
    project = Path(project).resolve()
    if board_size != 9 or not math.isfinite(komi) or gtp_timeout_seconds <= 0:
        raise ValueError('Invalid benchmark opponent board/komi/timeout.')
    config = types.SimpleNamespace(project=str(project),board_size=int(board_size),komi=float(komi),
                                   gtp_timeout_seconds=float(gtp_timeout_seconds),
                                   config_directory=str(config_directory or project / 'reports' / 'ahr_v2' / 'benchmark17_opponent_configs'))
    pin = go2_bench_opponent_pin(project, profile)
    if GO2_BENCH_PROFILES[profile]['engine'] == 'gnugo': return Go2BenchGNU(config,seed,pin)
    return Go2BenchKataGo(config,seed,pin)


In [ ]:
def go2_bench_validate_opponents(project, profiles=None, gtp_timeout_seconds=90, rules_namespace=None):
    project = Path(project).resolve()
    rules = rules_namespace or globals()
    assets = go2_bench_katago_assets(project, allow_download=False)
    profiles = list(profiles or GO2_BENCH_PROFILES)
    if any(profile not in GO2_BENCH_PROFILES for profile in profiles): raise ValueError('Unknown validation profile.')
    smoke_directory = project / 'reports' / 'ahr_v2' / 'fixtures' / 'benchmark17_opponent_smoke'
    smoke_directory.mkdir(parents=True,exist_ok=True)
    report = {'schema':'ahr-benchmark17-opponent-smoke-v1','assets':assets,'profiles':{},'all_passed':False}
    for profile in profiles:
        state = rules['go2_initial_state'](size=9,komi=7.5)
        trace = []
        with go2_bench_opponent_factory(project,profile,2026100917,config_directory=smoke_directory,gtp_timeout_seconds=gtp_timeout_seconds) as engine:
            runtime = engine.snapshot()
            for index in range(4):
                color = state.to_play
                if index == 0:
                    action = 40; engine.play(color,action); kind='play'
                else:
                    action = engine.genmove(color); kind='genmove'
                state = rules['go2_step'](state,action)
                trace.append({'kind':kind,'color':color,'action':action})
            original_next = engine.genmove(state.to_play)
            observed = engine.last_move_stats
        with go2_bench_opponent_factory(project,profile,2026100917,config_directory=smoke_directory,gtp_timeout_seconds=gtp_timeout_seconds) as restored:
            restored.replay(trace)
            replay_next = restored.genmove(state.to_play)
            assert replay_next == original_next, 'Fresh-process opponent replay differed.'
        mismatch_refused = False
        wrong_trace = [dict(item) for item in trace]
        wrong_trace[1]['action'] = (wrong_trace[1]['action'] + 1) % 82
        with go2_bench_opponent_factory(project,profile,2026100917,config_directory=smoke_directory,
                                       gtp_timeout_seconds=gtp_timeout_seconds) as rejected:
            try: rejected.replay(wrong_trace)
            except Go2OpponentError: mismatch_refused = True
        assert mismatch_refused, 'Incorrect opponent move history was silently accepted.'
        report['profiles'][profile] = {'trace':trace,'next_move':original_next,
                                       'fresh_process_replay_equal':True,'mismatched_replay_refused':True,'runtime':runtime,'last_move_stats':observed}
        print(profile, 'legal history and exact fresh-process replay passed', observed)
    report['all_passed'] = True
    path = project / 'reports' / 'ahr_v2' / 'fixtures' / 'benchmark17_opponents_validation.json'
    go2_atomic_json(path,report)
    print('Saved smoke evidence:',path)
    return report


In [ ]:
def go2_bench_validate_detached(project):
    """Prove detached fresh-kernel imports and frozen assets; no matches or training."""
    import nbformat, shlex, uuid
    project = Path(project).resolve()
    if not shutil.which('tmux'): raise FileNotFoundError('tmux dependency unavailable.')
    canonical = project / GO2_ROOT / GO2_BENCH_BOOK
    if not canonical.is_file(): raise FileNotFoundError('Canonical notebook 17 has not been created.')
    sources = go2_bench_sources(project)
    config = go2_bench_default_config()
    _, metadata = go2_bench_pinned_model(project, config, 'cpu')
    if metadata['optimizer_step'] != 8016:
        raise RuntimeError('Detached fixture expects the frozen step-8016 champion.')
    expected_player = {key:metadata[key] for key in ('run_id','optimizer_step','model_identity',
                                                    'checkpoint_file','checkpoint_sha256')}
    assets = go2_bench_katago_assets(project, allow_download=False)
    token = uuid.uuid4().hex[:12]
    session = 'ahr-v2-bench17-smoke-' + token
    directory = project / 'reports' / 'ahr_v2' / 'fixtures'
    directory.mkdir(parents=True, exist_ok=True)
    notebook = directory / ('benchmark17-detached-' + token + '.ipynb')
    receipt = directory / ('benchmark17-detached-' + token + '.json')
    log = directory / ('benchmark17-detached-' + token + '.log')
    lines = [
        'from pathlib import Path', 'import nbformat, json',
        'PROJECT = Path(' + repr(str(project)) + ')',
        'for name in ["00_contracts_and_recovery.ipynb","15_teacher_evaluation_and_launch.ipynb"]:',
        '    for cell in nbformat.read(PROJECT / "experiments/ahr_v2" / name, 4).cells:',
        '        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags",[]):',
        '            exec(compile(cell.source,name+":ahr-export","exec"),globals())',
        'receipt_path = Path(' + repr(str(receipt)) + ')',
        'try:',
        '    teacher = go2_teacher_arena_namespace(PROJECT)',
        '    globals().update(teacher)',
        '    for cell in nbformat.read(PROJECT / "experiments/ahr_v2" / ' + repr(GO2_BENCH_BOOK) + ',4).cells:',
        '        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags",[]):',
        '            exec(compile(cell.source,"17:ahr-export","exec"),globals())',
        '    torch.set_num_threads(4)',
        '    expected_sources = ' + repr(sources),
        '    observed_sources = go2_bench_sources(PROJECT)',
        '    assert observed_sources == expected_sources, "Detached worker source identity mismatch."',
        '    observed_assets = go2_bench_katago_assets(PROJECT,allow_download=False)',
        '    assert observed_assets == ' + repr(assets) + ', "Detached worker asset identity mismatch."',
        '    config = go2_bench_default_config()',
        '    model, metadata = go2_bench_pinned_model(PROJECT,config,"cpu")',
        '    player_pin = {key:metadata[key] for key in ("run_id","optimizer_step","model_identity","checkpoint_file","checkpoint_sha256")}',
        '    assert player_pin == ' + repr(expected_player) + ', "Detached champion identity mismatch."',
        '    assert metadata["optimizer_step"] == 8016 and not metadata["is_smoke"]',
        '    assert not model.training and all(not p.requires_grad for p in model.parameters())',
        '    pins = {profile["id"]:go2_bench_opponent_pin(PROJECT,profile["id"]) for profile in config["profiles"]}',
        '    report = {"status":"PASS","schema":"ahr-benchmark17-detached-fixture-v1",',
        '              "source_identity":observed_sources,"asset_identity":observed_assets,',
        '              "champion_pin":player_pin,"opponent_pins":pins,',
        '              "teacher_namespace_ready":all(key in teacher for key in ("go2_step","Go2Network","go2_search")),',
        '              "no_benchmark_or_training_launched":True,"completed_at_utc":go2_now()}',
        '    assert report["teacher_namespace_ready"]',
        '    go2_atomic_json(receipt_path,report)',
        '    print("PASS: detached setup and champion step 8016 verified; no matches or training.")',
        'except BaseException as error:',
        '    go2_atomic_json(receipt_path,{"status":"ERROR","error":repr(error),"source_identity":' + repr(sources) + '})',
        '    raise',
    ]
    book = nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell('# Detached notebook 17 setup validation\n\nDisposable fresh-kernel fixture. Expected runtime under 30 seconds. Imports tagged definitions, verifies source/asset/champion pins, writes a PASS receipt, and exits. No benchmark, training, experiment checkpoint or active-run pointer is created. Re-run the notebook 17 validation helper to generate another isolated fixture.'),
        nbformat.v4.new_code_cell('\n'.join(lines))],
        metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}})
    go2_atomic_bytes(notebook,nbformat.writes(book).encode())
    args = [str(project / '.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
            '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=25',str(notebook)]
    command = ' '.join(shlex.quote(argument) for argument in args) + ' >> ' + shlex.quote(str(log)) + ' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],
                   check=True,capture_output=True,text=True,timeout=10)
    deadline = time.monotonic() + 30
    result = None
    while time.monotonic() < deadline:
        alive = subprocess.run(['tmux','has-session','-t',session],capture_output=True,timeout=3).returncode == 0
        if receipt.is_file():
            result = json.loads(receipt.read_text())
            if result.get('status') != 'PASS' or not alive: break
        elif not alive: break
        time.sleep(0.2)
    alive = subprocess.run(['tmux','has-session','-t',session],capture_output=True,timeout=3).returncode == 0
    if alive:
        subprocess.run(['tmux','kill-session','-t',session],capture_output=True,timeout=3)
        raise TimeoutError('Disposable detached setup exceeded 30 seconds: ' + str(log))
    if result is None and receipt.is_file(): result = json.loads(receipt.read_text())
    if result is None or result.get('status') != 'PASS':
        detail = log.read_text()[-4000:] if log.is_file() else 'No worker log was produced.'
        raise RuntimeError('Disposable detached setup failed: ' + repr(result) + '\n' + detail)
    if result['source_identity'] != sources or result['champion_pin'] != expected_player or result['asset_identity'] != assets:
        raise RuntimeError('Detached validation receipt identity mismatch.')
    result.update(session=session,session_exited=True,fixture=str(notebook.relative_to(project)),
                  receipt=str(receipt.relative_to(project)),log=str(log.relative_to(project)))
    go2_atomic_json(project / 'reports/ahr_v2/benchmark17_detached_validation.json',result)
    return result


## Paired openings and timing

In [ ]:
import gzip, math

def go2_bench_protocol_identity(project, config, metadata, stage, extra=None):
    return {'stage':stage,'sources':go2_bench_sources(project),'config':copy.deepcopy(config),
            'model':{'run_id':metadata['run_id'],'checkpoint_file':metadata['checkpoint_file'],
                     'checkpoint_sha256':metadata['checkpoint_sha256'],'model_identity':metadata['model_identity']},
            'data':copy.deepcopy(extra or {})}

def go2_bench_protocol_checkpoint(model,identity,config,step,sampler,metrics,cuda_devices=()):
    return {'schema':GO2_SCHEMA,'identity':identity,'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
            'optimizer':None,'scheduler':None,'scaler':None,'rng':go2_capture_rng(cuda_devices),
            'sampler':copy.deepcopy(sampler),'global_step':int(step),'config':copy.deepcopy(config),'metrics':copy.deepcopy(metrics)}

def go2_bench_prepared_exclusions(project,metadata):
    cache_id=metadata['config']['cache_id']
    dataset=go2_teacher_dataset(project,cache_id,verify=False)
    train=set(dataset['manifest']['families']['train']);dev=set(dataset['manifest']['families']['dev'])
    if train&dev:raise ValueError('Prepared train/development opening families overlap.')
    consumed=set();files=[]
    for path in sorted((Path(project)/'reports/ahr_v2/evaluations').glob('*/report.json')):
        record=json.loads(path.read_text())
        if 'GNU' not in json.dumps(record.get('opponent',{})).upper() and record.get('config',{}).get('opponent')!='gnugo':
            # Current arena reports identify GNU Go inside opponent executable/version fields.
            if not any(token in json.dumps(record.get('opponent',{})).lower() for token in ('gnugo','gnu go')):continue
        accepted=[]
        for game in record.get('games',[]):
            actions=game.get('state',{}).get('actions',[])
            if len(actions)>=8:accepted.append(go2_teacher_opening_family(tuple(actions[:8])))
        if accepted:consumed.update(accepted);files.append({'file':str(path.relative_to(project)),'sha256':go2_file_sha(path),'families':len(set(accepted))})
    return {'train':train,'dev':dev,'consumed':consumed,'cache_identity':dataset['identity'],'prior_reviews':files}

def go2_bench_plan_report(config,metadata,sampler,identity,status):
    games=sampler['games'];families={g['family_sha'] for g in games if g['opening_actions']}
    return {'schema':'ahr-v2-benchmark-plan-v1','run_id':config['run_id'],'status':status,'config':copy.deepcopy(config),
            'model':copy.deepcopy(metadata),'identity':identity,'games':copy.deepcopy(games),
            'coverage':{'games':len(games),'balanced_pairs':len(games)//2,'fresh_opening_families':len(families),
                        'empty_board_pairs':sum(not g['opening_actions'] for g in games)//2,
                        'training_family_overlap':0,'development_family_overlap':0,'prior_review_family_overlap':0,
                        'attempts':sampler['attempts'],'opening_plies':config.get('opening_plies',8)},
            'test_payload_consumed':False,'provenance':'Frozen champion Scout sampled legal openings; no outcomes or winning labels.',
            'updated_at_utc':go2_now()}

def go2_bench_plan(project,config,model,metadata,max_pairs=None):
    project=go2_project(project);config=copy.deepcopy(config)
    if int(config.get('opening_plies',8))!=8:raise ValueError('Fresh-family benchmark requires exactly eight opening plies.')
    profiles=config['profiles']
    if not profiles or len({p['id'] for p in profiles})!=len(profiles):raise ValueError('Distinct opponent profiles required.')
    if any(int(p['games'])<2 or int(p['games'])%2 for p in profiles):raise ValueError('Each profile needs an even positive game count.')
    expected=sum(int(p['games']) for p in profiles);excluded=go2_bench_prepared_exclusions(project,metadata)
    extra={'cache_identity':excluded['cache_identity'],'excluded_train_sha':go2_digest(sorted(excluded['train'])),
           'excluded_dev_sha':go2_digest(sorted(excluded['dev'])),'excluded_prior_sha':go2_digest(sorted(excluded['consumed'])),
           'prior_reviews':excluded['prior_reviews']}
    identity=go2_bench_protocol_identity(project,config,metadata,'opening-plan',extra)
    run_dir=project/'checkpoints'/(config['run_id']+'-plan');report_path=project/'reports/ahr_v2/benchmarks'/config['run_id']/'plan.json'
    rng=np.random.default_rng(int(config['seed']));device=next(model.parameters()).device
    cuda=[int(device.index or 0)] if device.type=='cuda' else []
    with go2_checkpoint_lock(run_dir):
        saved,entry=go2_load_checkpoint(run_dir,identity)
        if saved is None:
            sampler={'games':[],'attempts':0,'profile_index':0,'pair_in_profile':0,'rng':copy.deepcopy(rng.bit_generator.state)};step=0
            go2_save_checkpoint(run_dir,go2_bench_protocol_checkpoint(model,identity,config,step,sampler,{},cuda))
        else:
            sampler=copy.deepcopy(saved['sampler']);step=saved['global_step'];rng.bit_generator.state=sampler['rng']
            if go2_tensor_digest(saved['model'])!=go2_tensor_digest(model.state_dict()):raise ValueError('Frozen opening model differs from checkpoint.')
            go2_restore_rng(saved['rng'])
        forbidden=excluded['train']|excluded['dev']|excluded['consumed']
        seen={g['family_sha'] for g in sampler['games'] if g['opening_actions']}
        committed_pairs=0;last_saved=time.monotonic()
        while sampler['profile_index']<len(profiles):
            if max_pairs is not None and committed_pairs>=int(max_pairs):break
            profile=profiles[sampler['profile_index']];pair_index=sampler['pair_in_profile']
            baseline=pair_index==0 and bool(config.get('empty_board_baseline',True))
            if baseline:actions=[];family=go2_teacher_opening_family(())
            else:
                accepted=False
                while sampler['attempts']<int(config.get('max_opening_attempts',10000)):
                    sampler['attempts']+=1;state=go2_initial_state(9,float(config.get('komi',7.5)));actions=[]
                    for ply in range(8):
                        out=go2_eval_batch(model,[state])[0];prior=np.asarray(out['prior'],dtype=np.float64)
                        legal=np.asarray(go2_legal_mask(state),bool);legal[-1]=False
                        temperature=float(config.get('opening_temperature',1.25));prior=np.where(legal,prior,0.)
                        prior=np.power(prior,1./temperature);mass=float(prior.sum())
                        if not np.isfinite(mass) or mass<=0:raise ValueError('Frozen Scout has no finite legal opening probability.')
                        action=int(rng.choice(len(prior),p=prior/mass));state=go2_step(state,action);actions.append(action)
                    family=go2_teacher_opening_family(tuple(actions))
                    if family not in forbidden and family not in seen:accepted=True;break
                    if time.monotonic()-last_saved>=120:
                        sampler['rng']=copy.deepcopy(rng.bit_generator.state);step+=1
                        go2_save_checkpoint(run_dir,go2_bench_protocol_checkpoint(model,identity,config,step,sampler,{'status':'GENERATING'},cuda));last_saved=time.monotonic()
                if not accepted:raise RuntimeError('Opening sampler exhausted its bound; no family reuse is permitted.')
                seen.add(family)
            pair_id=len(sampler['games'])//2;game_index=len(sampler['games']);opening_id=('empty-baseline' if baseline else 'opening-'+str(pair_id).zfill(3))
            for offset,color in enumerate((1,-1)):
                sampler['games'].append({'game_index':game_index+offset,'opponent_profile':profile['id'],'profile':profile['id'],
                    'opening_id':opening_id,'family_sha':family,'opening_family':family,'opening_actions':list(actions),
                    'ahr_color':color,'paired_game_index':game_index+1-offset,'pair_id':pair_id,
                    'seed':int(config['seed'])+pair_id,'opening_provenance':'empty-board reference' if baseline else 'frozen champion Scout, sampled without replacement by canonical D4 opening family'})
            sampler['pair_in_profile']+=1
            if sampler['pair_in_profile']>=int(profile['games'])//2:sampler['profile_index']+=1;sampler['pair_in_profile']=0
            sampler['rng']=copy.deepcopy(rng.bit_generator.state);step+=1;committed_pairs+=1
            metrics={'status':'COMPLETE' if len(sampler['games'])==expected else 'GENERATING','games':len(sampler['games'])}
            go2_save_checkpoint(run_dir,go2_bench_protocol_checkpoint(model,identity,config,step,sampler,metrics,cuda),best=metrics['status']=='COMPLETE')
            report=go2_bench_plan_report(config,metadata,sampler,identity,metrics['status']);go2_atomic_json(report_path,report);last_saved=time.monotonic()
        status='COMPLETE' if len(sampler['games'])==expected else 'PAUSED'
        report=go2_bench_plan_report(config,metadata,sampler,identity,status);go2_atomic_json(report_path,report)
    print('Opening plan:',report['coverage']['balanced_pairs'],'balanced pairs;',len(seen),'fresh canonical families; train/dev/prior review overlaps = 0.')
    return report

def go2_bench_validation_positions(project,count=12):
    """Timing only: never read a test payload, never generate a training label."""
    project=go2_project(project);root=project/'data/ahr_v2_teacher/raw'/GO2_TEACHER_COMMIT
    if not (root/'manifest.json').exists():raise FileNotFoundError('Pinned teacher source must already be prepared by notebook 12.')
    manifest=json.loads((root/'manifest.json').read_text());rows=[]
    for info in manifest['indices']:
        index=root/info['path']
        if go2_file_sha(index)!=info['sha256']:raise ValueError('Teacher source index changed.')
        with gzip.open(index,'rt') as handle:
            for line in handle:
                row=json.loads(line)
                if row['split']=='validation':rows.append(row)
    rows.sort(key=lambda row:row['game_id']);positions=[];rejected=0
    for row in rows:
        accepted=go2_teacher_convert_game(root,row,max_positions=3)
        if not accepted['accepted']:rejected+=1;continue
        metadata,arrays=go2_teacher_read_game(root,row);length=len(arrays['actions'])
        if length<16:continue
        picks=[min(8,length-2),length//2,length-3];stages=['opening','middle','late'];state=go2_initial_state(9,7.5)
        for ply,action in enumerate(arrays['actions']):
            if ply in picks and not state.terminal:
                stage=stages[picks.index(ply)]
                positions.append({'position_id':row['game_id']+'-'+str(ply),'stage':stage,'ply':ply,'state':go2_state_to_dict(state),
                                  'source_game_id':row['game_id'],'source_family':row['opening_family'],'source_split':'validation',
                                  'source_payload_sha256':row['sha256'],'purpose':'speed measurement only'})
            state=go2_step(state,int(action))
        if len(positions)>=int(count):break
    if len(positions)<int(count):raise RuntimeError('Not enough legally compatible validation games for complete speed coverage.')
    return positions[:int(count)],{'source_commit':GO2_TEACHER_COMMIT,'test_payload_consumed':False,'rejected_incompatible_games':rejected,
                                  'positions_sha256':go2_digest(positions[:int(count)]),'timing_positions_are_not_strength_trials':True}

def go2_bench_speed_summary(results):
    groups=[]
    keys=sorted({(r['device'],str(r['move_seconds'])) for r in results})
    for device,budget in keys:
        rows=[r for r in results if r['device']==device and str(r['move_seconds'])==budget];valid=[r for r in rows if r['status']=='MEASURED']
        elapsed=np.asarray([r['elapsed_seconds'] for r in valid],dtype=np.float64)
        visits=np.asarray([r['completed_simulations'] for r in valid],dtype=np.float64)
        groups.append({'device':device,'move_seconds':rows[0]['move_seconds'],'measured':len(valid),'errors':len(rows)-len(valid),
            'median_seconds':float(np.median(elapsed)) if len(elapsed) else None,
            'p95_seconds':float(np.quantile(elapsed,.95)) if len(elapsed) else None,'max_seconds':float(elapsed.max()) if len(elapsed) else None,
            'median_visits':float(np.median(visits)) if len(visits) else None,'min_visits':int(visits.min()) if len(visits) else None,
            'max_visits':int(visits.max()) if len(visits) else None,'overruns':sum(r['overrun_seconds']>0 for r in valid),
            'max_overrun_seconds':max((r['overrun_seconds'] for r in valid),default=None)})
    return groups

def go2_bench_speed(project,runid,config,model,metadata,plan,on_progress=None,max_probes=None):
    project=go2_project(project);config=copy.deepcopy(config)
    positions,timing_data=go2_bench_validation_positions(project,int(config.get('speed_positions',12)))
    identity=go2_bench_protocol_identity(project,config,metadata,'speed-probes',{'plan_sha256':go2_digest({'identity':plan['identity'],'games':plan['games'],'coverage':plan['coverage']}),'timing_data':timing_data})
    devices=list(config.get('speed_devices',['cpu','cuda:0']));budgets=list(config.get('speed_move_seconds',[1.,2.]))
    tasks=[{'device':device,'move_seconds':budget,'position_index':i} for device in devices for budget in budgets for i in range(len(positions))]
    run_dir=project/'checkpoints'/(runid+'-speed');report_path=project/'reports/ahr_v2/benchmarks'/runid/'speed.json'
    with go2_checkpoint_lock(run_dir):
        state,entry=go2_load_checkpoint(run_dir,identity)
        if state is None:
            sampler={'probe_cursor':0,'results':[],'cold_starts':[],'positions':positions,'tasks':tasks};step=0
            go2_save_checkpoint(run_dir,go2_bench_protocol_checkpoint(model,identity,config,step,sampler,{}))
        else:
            sampler=copy.deepcopy(state['sampler']);step=state['global_step'];go2_restore_rng(state['rng'])
            if go2_tensor_digest(state['model'])!=go2_tensor_digest(model.state_dict()):raise ValueError('Frozen timing model differs from checkpoint.')
        warmed={};processed=0
        while sampler['probe_cursor']<len(tasks):
            if max_probes is not None and processed>=int(max_probes):break
            task=tasks[sampler['probe_cursor']];device=task['device'];position=sampler['positions'][task['position_index']]
            if device not in warmed:
                initialized=torch.cuda.is_initialized();started=time.perf_counter()
                device_model=copy.deepcopy(model).to(device);device_model.eval();device_model.requires_grad_(False)
                if str(device).startswith('cuda'):torch.cuda.synchronize(torch.device(device))
                transfer_seconds=time.perf_counter()-started;started=time.perf_counter()
                go2_eval_batch(device_model,[go2_state_from_dict(position['state'])])
                if str(device).startswith('cuda'):torch.cuda.synchronize(torch.device(device))
                sampler['cold_starts'].append({'device':device,'process_cuda_initialized_before':initialized,'model_transfer_seconds':transfer_seconds,
                                              'first_forward_seconds':time.perf_counter()-started,'at_probe_cursor':sampler['probe_cursor'],
                                              'note':'Process startup measurement; subsequent probes use this warmed device model.'})
                warmed[device]=device_model
            device_model=warmed[device];budget=task['move_seconds'];state_obj=go2_state_from_dict(position['state'])
            search=Go2SearchConfig(simulations=int(config.get('speed_max_simulations',512)),batch_size=int(config.get('batch_size',8)),
                                  wall_time_seconds=budget,seed=int(config['seed'])+task['position_index'],root_noise_fraction=0.,
                                  reading_wall_time_seconds=config.get('speed_reading_wall_time_seconds',.05))
            if str(device).startswith('cuda'):torch.cuda.synchronize(torch.device(device))
            started=time.perf_counter();out=None
            try:
                out=go2_search(device_model,state_obj,search,model_identity=metadata['model_identity'],project=project,
                               context={'kind':'benchmark_speed','run_id':runid,'position_id':position['position_id']})
                if str(device).startswith('cuda'):torch.cuda.synchronize(torch.device(device))
                elapsed=time.perf_counter()-started
                if int(out['action']) not in go2_legal_actions(state_obj):raise AssertionError('Speed probe selected an illegal move.')
                row={**task,'position_id':position['position_id'],'stage':position['stage'],'ply':position['ply'],'status':'MEASURED',
                     'elapsed_seconds':elapsed,'completed_simulations':int(out['completed_simulations']),'action':int(out['action']),
                     'search_status':out['status'],'reading_nodes':int(out.get('reading_nodes',0)),
                     'overrun_seconds':max(0.,elapsed-float(budget)) if budget is not None else 0.,
                     'measurement_includes':'Features, legality/replay checks, neural evaluation, tactical reading, search, and CUDA synchronization.'}
            except Exception as error:
                if str(device).startswith('cuda'):torch.cuda.synchronize(torch.device(device))
                row={**task,'position_id':position['position_id'],'stage':position['stage'],'ply':position['ply'],'status':'ERROR',
                     'elapsed_seconds':time.perf_counter()-started,'completed_simulations':None,'action':None,'error':type(error).__name__+': '+str(error)}
            sampler['results'].append(row);sampler['probe_cursor']+=1;processed+=1;step+=1
            cuda=sorted({int(torch.device(d).index or 0) for d in warmed if str(d).startswith('cuda')})
            metrics={'status':'COMPLETE' if sampler['probe_cursor']==len(tasks) else 'MEASURING','probe_cursor':sampler['probe_cursor'],'summary':go2_bench_speed_summary(sampler['results'])}
            go2_save_checkpoint(run_dir,go2_bench_protocol_checkpoint(model,identity,config,step,sampler,metrics,cuda),best=metrics['status']=='COMPLETE')
            report={'schema':'ahr-v2-benchmark-speed-v1','run_id':runid,'status':metrics['status'],'identity':identity,
                    'planned_probes':len(tasks),'completed_probes':sampler['probe_cursor'],'cold_starts':sampler['cold_starts'],
                    'results':sampler['results'],'summary':metrics['summary'],'timing_data':timing_data,
                    'deadline_note':'A search time target is soft. Atomic batches, replay checks, and tactical reading can overrun; observed p95 is not a hard real-time guarantee.',
                    'updated_at_utc':go2_now()}
            go2_atomic_json(report_path,report)
            if on_progress is not None:on_progress(copy.deepcopy(report))
        report={'schema':'ahr-v2-benchmark-speed-v1','run_id':runid,'status':'COMPLETE' if sampler['probe_cursor']==len(tasks) else 'PAUSED','identity':identity,
                'planned_probes':len(tasks),'completed_probes':sampler['probe_cursor'],'cold_starts':sampler['cold_starts'],
                'results':sampler['results'],'summary':go2_bench_speed_summary(sampler['results']),'timing_data':timing_data,
                'deadline_note':'Soft search targets; observed medians/p95/max include replay, features, reading, search, and CUDA synchronization. Completed samples are retained exactly on restart; interrupted probes are rerun with new wall-clock measurements.',
                'updated_at_utc':go2_now()}
        go2_atomic_json(report_path,report)
    return report


def go2_bench_protocol_scientific(report):
    return [{key:row.get(key) for key in ('device','move_seconds','position_id','stage','ply','status','completed_simulations','action','search_status','reading_nodes')} for row in report['results']]

def go2_bench_protocol_fixture_worker(project,fixture,phase,label):
    project=go2_project(project);path=project/'reports/ahr_v2/fixtures'/('benchmark17-protocol-'+label+'.ipynb')
    canonical=project/GO2_ROOT/'17_strength_and_speed_benchmarks.ipynb'
    source=canonical if canonical.exists() else project/'reports/ahr_v2/fixtures/benchmark17_protocol.ipynb'
    setup="from pathlib import Path\nimport nbformat\nPROJECT=Path("+repr(str(project))+ ")\nns={'PROJECT':PROJECT}\nbase=nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4)\nfor cell in base.cells:\n    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,ns)\nns['go2_import_exports'](PROJECT/'experiments/ahr_v2/15_teacher_evaluation_and_launch.ipynb',ns)\nns.update(ns['go2_teacher_arena_namespace'](PROJECT))\nns['go2_import_exports'](Path("+repr(str(source))+"),ns)\n"
    if source!=canonical:
        setup+="ns['go2_bench_sources']=lambda project:{'protocol_draft':ns['go2_code_sha'](PROJECT/'reports/ahr_v2/fixtures/benchmark17_protocol.ipynb'),**ns['go2_teacher_control_sources'](PROJECT)}\n"
    setup+="import torch\ntorch.set_num_threads(4)\nmodel,metadata=ns['go2_load_selected_teacher_player'](PROJECT,device='cpu')\nfixture="+repr(fixture)+"\n"
    maximum=2 if phase=='pause' else None
    body="plan=ns['go2_bench_plan'](PROJECT,fixture,model,metadata,max_pairs="+repr(maximum)+")\n"
    if phase=='speed-pause':
        body="plan=ns['go2_bench_plan'](PROJECT,fixture,model,metadata)\n"+"speed=ns['go2_bench_speed'](PROJECT,fixture['run_id'],fixture,model,metadata,plan,max_probes=1)\n"
    elif phase=='speed-resume':
        body+="speed=ns['go2_bench_speed'](PROJECT,fixture['run_id'],fixture,model,metadata,plan)\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Fresh kernel checkpoint fixture; no full benchmark or model training.'),nbformat.v4.new_code_cell(setup+body)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(nb,path)
    result=subprocess.run([str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=240',str(path)],cwd=project,capture_output=True,text=True,timeout=300)
    if result.returncode:raise RuntimeError('Fresh protocol fixture failed: '+result.stdout[-3000:]+result.stderr[-6000:])
    return str(path.relative_to(project))

def go2_bench_protocol_validation(project):
    project=go2_project(project);torch.set_num_threads(4);model,metadata=go2_load_selected_teacher_player(project,device='cpu')
    token=uuid.uuid4().hex[:8]
    config={'run_id':'benchmark17-protocol-'+token+'-full','seed':2026100917,'board_size':9,'komi':7.5,
            'profiles':[{'id':'gnugo10','kind':'gnugo','level':10,'games':8}], 'opening_plies':8,'opening_temperature':1.25,
            'empty_board_baseline':True,'max_opening_attempts':128,'speed_positions':3,'speed_devices':['cpu'],
            'speed_move_seconds':[None],'speed_max_simulations':8,'batch_size':2,'speed_reading_wall_time_seconds':None}
    full=go2_bench_plan(project,config,model,metadata)
    resumed_config={**config,'run_id':'benchmark17-protocol-'+token+'-resume'}
    notebooks=[go2_bench_protocol_fixture_worker(project,resumed_config,'pause',token+'-plan-pause'),
               go2_bench_protocol_fixture_worker(project,resumed_config,'resume',token+'-plan-resume')]
    resumed=json.loads((project/'reports/ahr_v2/benchmarks'/resumed_config['run_id']/'plan.json').read_text())
    if full['games']!=resumed['games']:raise AssertionError('Fresh-kernel opening RNG/cursor recovery differs.')
    excluded=go2_bench_prepared_exclusions(project,metadata)
    families={g['family_sha'] for g in full['games'] if g['opening_actions']}
    if families&(excluded['train']|excluded['dev']|excluded['consumed']):raise AssertionError('Benchmark opening leakage.')
    for first,second in zip(full['games'][::2],full['games'][1::2]):
        if first['opening_actions']!=second['opening_actions'] or (first['ahr_color'],second['ahr_color'])!=(1,-1):raise AssertionError('Color pair imbalance.')
        state=go2_initial_state(9,7.5)
        for action in first['opening_actions']:state=go2_step(state,action)
    wrong={**resumed_config,'seed':resumed_config['seed']+1}
    try:go2_bench_plan(project,wrong,model,metadata)
    except RuntimeError:pass
    else:raise AssertionError('Opening recipe identity mismatch was accepted.')
    speed=go2_bench_speed(project,config['run_id'],config,model,metadata,full)
    notebooks.extend([go2_bench_protocol_fixture_worker(project,resumed_config,'speed-pause',token+'-speed-pause'),
                      go2_bench_protocol_fixture_worker(project,resumed_config,'speed-resume',token+'-speed-resume')])
    recovered=json.loads((project/'reports/ahr_v2/benchmarks'/resumed_config['run_id']/'speed.json').read_text())
    if go2_bench_protocol_scientific(speed)!=go2_bench_protocol_scientific(recovered):raise AssertionError('Clockless speed search choices differ after fresh-kernel recovery.')
    if any(r['status']!='MEASURED' or r['completed_simulations']!=8 for r in speed['results']):raise AssertionError('Clockless probes failed or lost visits.')
    report={'schema':'ahr-v2-benchmark-protocol-validation-v1','status':'PASS','checked_at_utc':go2_now(),
            'opening_fresh_kernel_exact':True,'speed_clockless_fresh_kernel_exact':True,'legal_openings':True,
            'balanced_color_pairs':True,'train_dev_prior_review_disjoint':True,'mismatched_recipe_refused':True,
            'test_payload_consumed':False,'fresh_fixtures':notebooks,'speed_positions_by_stage':[r['stage'] for r in speed['results']],
            'fixture_runs':[config['run_id'],resumed_config['run_id']],'note':'Wall-clock durations are measurements and deliberately excluded from exact scientific-state comparison.'}
    go2_atomic_json(project/'reports/ahr_v2/benchmark17_protocol_validation.json',report)
    return report


## Resumable arena

In [ ]:
class _Go2BenchPause(Exception):
    pass


def _go2_bench_clock_sync(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize(torch.device(device))


def _go2_bench_cuda_devices(device):
    if not str(device).startswith('cuda'):return []
    return [torch.device(device).index if torch.device(device).index is not None else torch.cuda.current_device()]


def _go2_bench_game_rows(plan):
    rows=plan.get('games',[]) if isinstance(plan,dict) else plan
    if not isinstance(rows,list) or not rows:raise ValueError('A nonempty explicit paired game plan is required.')
    if [r['game_index'] for r in rows]!=list(range(len(rows))):raise ValueError('Game indices must be consecutive.')
    return rows


def _go2_bench_plan_pin(plan):
    return {k:copy.deepcopy(plan[k]) for k in ['identity','games','coverage'] if k in plan} if isinstance(plan,dict) else copy.deepcopy(plan)


def _go2_bench_profile(row):
    return row.get('opponent_profile',row.get('profile'))


def _go2_bench_summary(games):
    counts={key:sum(g['outcome']==key for g in games) for key in ['WIN','LOSS','DRAW','UNKNOWN']}
    n=counts['WIN']+counts['LOSS']+counts['DRAW'];wins=counts['WIN']
    z=1.959963984540054
    if n:
        p=wins/n;den=1+z*z/n;center=(p+z*z/(2*n))/den
        radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
        interval=[max(0,center-radius),min(1,center+radius)]
    else:p=None;interval=None
    moves=[t for g in games for t in g['trace'] if t.get('kind')=='play' and t.get('ahr_move')]
    times=np.asarray([m['ahr_move_seconds'] for m in moves],dtype=np.float64)
    return {'attempts':len(games),'completed':n,**counts,'win_rate_completed':p,
        'win_rate_all_attempts_lower':wins/len(games) if games else None,
        'wilson_95':interval,'wilson_95_note':'Assumes independent Bernoulli games. Paired openings violate this assumption; descriptive only, not rank or strength certification.','ahr_moves':len(moves),
        'timing':None if not len(times) else {'mean':float(times.mean()),'p50':float(np.quantile(times,.5)),
        'p95':float(np.quantile(times,.95)),'p99':float(np.quantile(times,.99)),'maximum':float(times.max()),
        'budget_overruns':sum(bool(m['budget_overrun']) for m in moves),
        'budget_overrun_rate':sum(bool(m['budget_overrun']) for m in moves)/len(moves),
        'simulations_mean':float(np.mean([m['search_simulations'] for m in moves])),
        'policy_support_mean':float(np.mean([m['policy_support_simulations'] for m in moves])),
        'zero_simulation_moves':sum(m['search_simulations']==0 for m in moves),
        'reading_nodes_mean':float(np.mean([m['reading_nodes'] for m in moves]))}}


def _go2_bench_report(state,project,run_id,status):
    profiles=sorted({_go2_bench_profile(row) for row in _go2_bench_game_rows(state['plan'])})
    report={'schema':'ahr-v2-strength-speed-benchmark-v1','run_id':run_id,'status':status,
        'purpose':state['config']['benchmark']['purpose'],'identity':copy.deepcopy(state['identity']),
        'config':copy.deepcopy(state['config']['benchmark']),'plan':copy.deepcopy(state['plan']),
        'summary':_go2_bench_summary(state['replay']),
        'by_profile':{profile:_go2_bench_summary([g for g in state['replay'] if g['profile']==profile]) for profile in profiles},
        'games':copy.deepcopy(state['replay']),'next_game_index':state['sampler']['game_index'],
        'active_ply':len(state['self_play']['trace']) if state['self_play'] else None,
        'active_profile':state['self_play']['profile'] if state['self_play'] else None,
        'global_event':state['global_step'],'updated_at_utc':go2_now(),
        'timing_scope':'End-to-end AHR decision, including CUDA synchronization, encoding, root tactical work, bounded reading and model/search work. Soft time budget; atomic work may overrun.',
        'recovery_scope':'Fixed-simulation, clockless fixtures restore exactly. Wall-clock budgets preserve consumed time and state; subsequent scheduling can change achieved visits.',
        'strength_scope':'Development benchmark, not rank certification. UNKNOWN attempts are excluded from completed-game win rates and explicitly counted.',
        'resume_cell':'17: Run All with the same frozen benchmark configuration and run_id.'}
    go2_atomic_json(Path(project)/'reports/ahr_v2/benchmarks'/run_id/'report.json',report)
    return report


def go2_bench_arena(project,run_id,config,plan,model,metadata,opponent_factory,on_progress=None,max_events=None,*,_search_evaluator=None):
    """Pinned development arena. Reads teacher weights; never trains or consumes training labels."""
    project=go2_project(project);config=copy.deepcopy(config);rows=_go2_bench_game_rows(plan)
    if Path(run_id).name!=run_id or not run_id.startswith('ahr-v2-'):raise ValueError('Local ahr-v2- run ID required.')
    purpose=config.get('purpose','development')
    if purpose not in ['development','validation_fixture']:raise ValueError('Only development and disposable validation purposes are allowed.')
    config['purpose']=purpose
    if _search_evaluator is not None and purpose!='validation_fixture':raise ValueError('Injected evaluators require disposable validation_fixture purpose.')
    if purpose=='development' and (metadata.get('is_smoke') or metadata.get('optimizer_step',0)<=0):raise ValueError('A trained nonsmoke champion is required.')
    board_size=int(config.get('board_size',9));komi=float(config.get('komi',7.5));device=config.get('device','cuda:0')
    if config.get('ahr_move_seconds') is not None and config['ahr_move_seconds']<=0:raise ValueError('Positive soft move budget or None required.')
    if config.get('max_simulations',512)<1 or config.get('batch_size',8)<1:raise ValueError('Positive search budget required.')
    model=model.to(device).eval();model.requires_grad_(False);model_id=go2_model_identity(model)
    if metadata.get('model_identity',model_id)!=model_id:raise RuntimeError('Loaded model identity differs from checkpoint metadata.')
    profiles=sorted({_go2_bench_profile(row) for row in rows})
    if None in profiles:raise ValueError('Every planned game needs a profile.')
    pins={p:go2_bench_opponent_pin(project,p) for p in profiles}
    identity={'schema':'ahr-v2-frozen-strength-speed-arena-v1','sources':go2_bench_sources(project),
        'model_identity':model_id,'training_run_id':metadata['run_id'],
        'checkpoint':{'file':metadata['checkpoint_file'],'sha256':metadata['checkpoint_sha256'],
                      'optimizer_step':metadata.get('optimizer_step',metadata.get('global_step',0))},
        'training_sources':copy.deepcopy(metadata.get('sources',{})),'plan_sha256':go2_digest(_go2_bench_plan_pin(plan)),
        'benchmark':copy.deepcopy(config),'opponents':pins,'network':dataclasses.asdict(model.config)}
    directory=project/'checkpoints'/run_id
    with go2_checkpoint_lock(directory):
        state,entry=go2_load_checkpoint(directory,identity=identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':{},'scheduler':{},'scaler':{},'rng':go2_capture_rng(_go2_bench_cuda_devices(device)),
                'sampler':{'game_index':0},'global_step':0,'config':{'benchmark':config,'network':dataclasses.asdict(model.config)},
                'metrics':{'completed_games':0,'unknown_games':0},'plan':copy.deepcopy(plan),'replay':[],
                'self_play':None,'search':None,'distributed':{'kind':'single_writer_single_actor'},'data_cursor':{'game_index':0},
                'evaluation_only':True}
            go2_save_checkpoint(directory,state,best=True,resume_cell='17: Run All with identical frozen benchmark configuration.')
        else:
            if go2_tensor_digest(state['model'])!=go2_tensor_digest(model.state_dict()):raise RuntimeError('Arena model snapshot hash mismatch.')
            model.load_state_dict(state['model']);go2_restore_rng(state['rng'])
        event_start=state['global_step'];last_checkpoint_event=state['global_step'];last_checkpoint_time=time.monotonic()
        live_tree=None;game=None;opponent=None;game_timer=None;game_elapsed_base=0.0
        report=_go2_bench_report(state,project,run_id,'RUNNING' if state['sampler']['game_index']<len(rows) else 'COMPLETE')
        def capture_elapsed():
            if state['self_play'] is None:return
            if game_timer is not None:state['self_play']['elapsed_seconds']=game_elapsed_base+time.monotonic()-game_timer
        def save(force=False,notify=True):
            nonlocal last_checkpoint_time,last_checkpoint_event,game_timer,game_elapsed_base
            due=force or time.monotonic()-last_checkpoint_time>=120 or state['global_step']-last_checkpoint_event>=250
            if not due:return
            capture_elapsed()
            if game is not None and state['self_play'] is not None:state['self_play']['state']=go2_state_to_dict(game)
            state['search']=go2_search_state_dict(live_tree) if live_tree is not None else None
            state['rng']=go2_capture_rng(_go2_bench_cuda_devices(device));state['metrics']={'completed_games':sum(g['outcome']!='UNKNOWN' for g in state['replay']),
                'unknown_games':sum(g['outcome']=='UNKNOWN' for g in state['replay'])}
            state['data_cursor']['game_index']=state['sampler']['game_index']
            if state['global_step']<=last_checkpoint_event:state['global_step']=last_checkpoint_event+1
            go2_save_checkpoint(directory,state,resume_cell='17: Run All with identical frozen benchmark configuration.')
            last_checkpoint_event=state['global_step'];last_checkpoint_time=time.monotonic()
            if state['self_play'] is not None:
                game_elapsed_base=state['self_play']['elapsed_seconds'];game_timer=time.monotonic()
            status='COMPLETE' if state['sampler']['game_index']>=len(rows) else 'RUNNING'
            report=_go2_bench_report(state,project,run_id,status)
            if notify and on_progress is not None:on_progress(report)
        def event(force=False,allow_save=True):
            state['global_step']+=1
            if max_events is not None and state['global_step']-event_start>=max_events:
                save(force=True);raise _Go2BenchPause()
            if allow_save:save(force=force)
        def on_batch(tree):
            nonlocal live_tree
            live_tree=tree
            if state['self_play'].get('move_started') is not None:
                state['self_play']['move_elapsed_seconds']=state['self_play']['move_elapsed_base']+time.monotonic()-state['self_play']['move_started']
            event(allow_save=False)
        try:
            while state['sampler']['game_index']<len(rows):
                index=state['sampler']['game_index'];row=rows[index];profile=_go2_bench_profile(row)
                if int(row['ahr_color']) not in [-1,1]:raise ValueError('AHR color must be Black 1 or White -1.')
                if state['self_play'] is None:
                    game=go2_initial_state(board_size,komi)
                    opening_trace=[]
                    for action in row.get('opening_actions',[]):
                        color=game.to_play;game=go2_step(game,int(action))
                        opening_trace.append({'kind':'play','color':color,'action':int(action),'opening_forced':True,'ahr_move':False})
                    if game.terminal:raise ValueError('Opening must leave a nonterminal legal game.')
                    seed=int(row.get('seed',config.get('seed',2026100917)+min(index,int(row.get('paired_game_index',index)))))
                    state['self_play']={'state':go2_state_to_dict(game),'trace':opening_trace,'profile':profile,'seed':seed,
                        'ahr_color':int(row['ahr_color']),'elapsed_seconds':0.0,'move_elapsed_seconds':0.0,
                        'move_elapsed_base':0.0,'move_started':None,'opponent_snapshot':None}
                    active=state['self_play'];live_tree=None;game_timer=None;event(force=True)
                else:game=go2_state_from_dict(state['self_play']['state']);go2_verify_replay(game)
                active=state['self_play'];color=active['ahr_color'];game_elapsed_base=active['elapsed_seconds'];game_timer=None
                unknown_reason=None;opponent=None
                try:
                    opponent=opponent_factory(profile,active['seed'],board_size,komi)
                    if hasattr(opponent,'pin') and go2_digest(opponent.pin)!=go2_digest(pins[profile]):raise RuntimeError('Opponent factory pin differs from frozen benchmark identity.')
                    # Reconstruct external engine from a fixed seed and verify all prior generated moves.
                    opponent.replay(active['trace'])
                    game_timer=time.monotonic()
                    while not game.terminal:
                        elapsed=game_elapsed_base+time.monotonic()-game_timer
                        if game.move_number>=int(config.get('max_plies',600)):
                            unknown_reason='PLY_CUTOFF';break
                        if config.get('game_time_seconds',900) is not None and elapsed>=config.get('game_time_seconds',900):
                            unknown_reason='GAME_TIMEOUT';break
                        if game.to_play==color:
                            active['move_elapsed_base']=active.get('move_elapsed_seconds',0.0) if state['search'] is not None else 0.0
                            active['move_started']=time.monotonic();_go2_bench_clock_sync(device)
                            search_config=Go2SearchConfig(simulations=int(config.get('max_simulations',512)),batch_size=int(config.get('batch_size',8)),
                                seed=active['seed']*10000+game.move_number,wall_time_seconds=config.get('ahr_move_seconds',2.0),
                                reading_wall_time_seconds=config.get('reading_wall_time_seconds',0.05),
                                reading_total_nodes=config.get('reading_total_nodes',256),reading_quantum=config.get('reading_quantum',16))
                            context={'purpose':purpose,'benchmark_id':run_id,'game_index':index,'ply':game.move_number,'profile':profile}
                            live_tree=None if state['search'] is None else go2_search_from_state_dict(state['search'],model,game,search_config,context=context,project=project)
                            output=go2_search(model,game,search_config,tree=live_tree,context=context,project=project,
                                evaluator=_search_evaluator,on_batch=on_batch)
                            live_tree=output['tree'];_go2_bench_clock_sync(device)
                            seconds=active['move_elapsed_base']+time.monotonic()-active['move_started']
                            action=int(output['action']);successor=go2_step(game,action)
                            trace_item={'kind':'play','color':game.to_play,'action':action,'ahr_move':True,
                                'ahr_move_seconds':seconds,'soft_budget_seconds':config.get('ahr_move_seconds',2.0),
                                'budget_overrun':config.get('ahr_move_seconds',2.0) is not None and seconds>config.get('ahr_move_seconds',2.0),
                                'search_simulations':int(output['completed_simulations']),
                                'policy_support_simulations':int(sum(output.get('policy_visit_counts',output['visit_counts']))),
                                'verification_simulations':int(sum(output.get('verification_visit_counts',[]))),
                                'search_status':output['search_status'],'search_completion':output['status'],
                                'reading_nodes':int(output.get('reading_nodes',0)),'reading_pending':int(output.get('reading_pending',0)),
                                'reading_statuses':{key:sum(f['status']==key for f in output.get('reading_facts',[])) for key in [GO2_READING_PROVED,GO2_READING_REFUTED,'UNKNOWN']},
                                'counters':copy.deepcopy(output['counters'])}
                            # External play lies outside AHR decision latency, but inside the game clock.
                            opponent.play(game.to_play,action)
                        else:
                            action=int(opponent.genmove(game.to_play));successor=go2_step(game,action)
                            trace_item={'kind':'genmove','color':game.to_play,'action':action,'ahr_move':False,
                                'opponent_stats':copy.deepcopy(getattr(opponent,'last_move_stats',{}))}
                        active['trace'].append(trace_item);game=successor;active['state']=go2_state_to_dict(game)
                        active['move_elapsed_seconds']=0.0;active['move_elapsed_base']=0.0;active['move_started']=None
                        live_tree=None;state['search']=None
                        if hasattr(opponent,'snapshot'):active['opponent_snapshot']=opponent.snapshot()
                        event()
                        if config.get('game_time_seconds',900) is not None and game_elapsed_base+time.monotonic()-game_timer>=config.get('game_time_seconds',900):
                            unknown_reason='GAME_TIMEOUT';break
                except _Go2BenchPause:raise
                except (TimeoutError,Go2IllegalMove) as exc:
                    unknown_reason=type(exc).__name__+': '+str(exc)[:250]
                except Go2OpponentError as exc:
                    if 'regenerated a different' in str(exc) or 'replay mismatch' in str(exc).lower():raise RuntimeError('Opponent recovery is nondeterministic; resume refused.') from exc
                    unknown_reason='OPPONENT_FAILURE: '+str(exc)[:250]
                finally:
                    if opponent is not None:opponent.close()
                    opponent=None
                capture_elapsed()
                # Time and ply cutoffs stay UNKNOWN even when their last move happens to end a game.
                if unknown_reason is None and game.terminal:
                    result=go2_outcome(game);winner=result['winner'];label='DRAW' if winner==0 else 'WIN' if winner==color else 'LOSS'
                    score=color*result['score_black']
                else:label='UNKNOWN';score=None
                finished={'game_index':index,'profile':profile,'ahr_color':color,'opponent_seed':active['seed'],
                    'opening_actions':row.get('opening_actions',[]),'opening_id':row.get('opening_id'),
                    'opening_family':row.get('family_sha',row.get('opening_family')),'paired_game_index':row.get('paired_game_index'),
                    'outcome':label,'unknown_reason':unknown_reason,'plies':game.move_number,'score_for_ahr':score,
                    'elapsed_seconds':active['elapsed_seconds'],'state':go2_state_to_dict(game),'trace':copy.deepcopy(active['trace'])}
                state['replay'].append(finished);state['sampler']['game_index']+=1
                state['self_play']=None;state['search']=None;live_tree=None;game=None;game_timer=None;event(force=True)
        except _Go2BenchPause:
            return _go2_bench_report(state,project,run_id,'PAUSED')
        except BaseException:
            # Preserve the precise live stack, pending inference, game and RNG before surfacing an interruption.
            if state['self_play'] is not None and active.get('move_started') is not None:
                active['move_elapsed_seconds']=active['move_elapsed_base']+time.monotonic()-active['move_started']
                active['move_started']=None
            save(force=True,notify=False)
            raise
        finally:
            if opponent is not None:opponent.close()
        return _go2_bench_report(state,project,run_id,'COMPLETE')


In [ ]:
def go2_bench_arena_validation(project=None,*,_export_path=None):
    """Short disposable 3x3 fixture, fresh-kernel exact pending inference/reader recovery, no strength match."""
    import subprocess,sys
    project=go2_project(project)
    export_path=Path(_export_path) if _export_path is not None else project/GO2_ROOT/'17_strength_and_speed_benchmarks.ipynb'
    if not export_path.is_file():export_path=project/'reports/ahr_v2/fixtures/benchmark17_arena.ipynb'
    namespace=globals().copy()
    # Isolate injected fixtures from the production factory and namespace.
    go2_import_exports(export_path,namespace)
    helpers=r"""def go2_bench_opponent_pin(project,profile):
    return {'name':'disposable_pass_fixture','profile':profile,'sha256':go2_digest('pass_fixture17v1')}
class _Go2BenchPassFixture:
    def __init__(self,profile,seed,size,komi):self.pin=go2_bench_opponent_pin(None,profile);self.size=size;self.trace=[]
    def play(self,color,action):self.trace.append((color,action))
    def genmove(self,color):self.trace.append((color,self.size*self.size));return self.size*self.size
    def replay(self,trace):
        for item in trace:
            if item['kind']=='play':self.play(item['color'],item['action'])
            elif self.genmove(item['color'])!=item['action']:raise Go2OpponentError('regenerated a different fixture action')
    def close(self):pass
    def snapshot(self):return {'trace':list(self.trace)}
def _go2_bench_fixture_evaluator(states):
    out=[]
    for state in states:
        prior=np.zeros(state.size*state.size+1);prior[state.pass_action]=1.0
        out.append({'prior':prior,'value':0.,'score':0.})
    return out
"""
    exec(helpers,namespace)
    arena=namespace['go2_bench_arena'];factory=namespace['_Go2BenchPassFixture'];evaluator=namespace['_go2_bench_fixture_evaluator']
    model=Go2Network(Go2NetConfig(board_size=3,width=16,blocks=1)).eval()
    metadata={'run_id':'ahr-v2-disposable-fixture','checkpoint_file':'disposable','checkpoint_sha256':go2_tensor_digest(model.state_dict()),
        'optimizer_step':0,'is_smoke':True,'model_identity':go2_model_identity(model)}
    cfg={'purpose':'validation_fixture','device':'cpu','board_size':3,'komi':0.,'ahr_move_seconds':None,'max_simulations':8,'batch_size':2,
        'max_plies':12,'game_time_seconds':None,'reading_wall_time_seconds':None,'reading_total_nodes':16,'reading_quantum':1,'seed':1701}
    rows=[{'game_index':i,'profile':'pass-fixture','opening_actions':[0,3,1,4],'ahr_color':1 if i==0 else -1,
        'seed':1701,'paired_game_index':1-i} for i in range(2)]
    plan={'identity':{'purpose':'validation_fixture'},'games':rows,'coverage':{'pairs':1},'updated_at_utc':go2_now()}
    prefix='ahr-v2-bench17-validation-'+uuid.uuid4().hex[:8]
    full=arena(project,prefix+'-full',cfg,plan,model,metadata,factory,_search_evaluator=evaluator)
    paused=arena(project,prefix+'-resume',cfg,plan,model,metadata,factory,max_events=2,_search_evaluator=evaluator)
    saved,_=go2_load_checkpoint(project/'checkpoints'/(prefix+'-resume'))
    assert paused['status']=='PAUSED' and saved['search']['completed_simulations']==2
    root=go2_state_from_dict(saved['self_play']['state'])
    context={'purpose':'validation_fixture','benchmark_id':prefix+'-resume','game_index':0,'ply':4,'profile':'pass-fixture'}
    tree=go2_search_from_state_dict(saved['search'],model,root,context=context,project=project)
    _go2_collect_batch(tree,Go2SearchConfig(**tree['config']),_go2_search_game(),2)
    saved['search']=go2_search_state_dict(tree);saved['global_step']+=1
    checks={'pending_reader_and_inference':bool(tree['pending']) and any(j['status']=='PENDING' for j in tree['reading_jobs'])}
    with go2_checkpoint_lock(project/'checkpoints'/(prefix+'-resume')):go2_save_checkpoint(project/'checkpoints'/(prefix+'-resume'),saved)
    path=project/'reports/ahr_v2/fixtures'/('benchmark17-fresh-validation-'+prefix[-8:]+'.ipynb');result_path=path.with_suffix('.json')
    # Use the same immutable canonical exports in a genuinely fresh project kernel.
    setup="from pathlib import Path\nimport nbformat\nproject=Path("+repr(str(project))+ ")\n"
    names=['00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb',
           '05_evaluation_and_progress.ipynb','08_tactical_constraints_and_search.ipynb','12_teacher_data_and_targets.ipynb',
           '13_teacher_pretraining.ipynb','14_worker_reading_and_search.ipynb','15_teacher_evaluation_and_launch.ipynb']
    # Exact current notebook names are discovered because source identities still verify their contents.
    for pref in ['12','13']:
        names=[name for name in names if not name.startswith(pref)]
        found=list((project/GO2_ROOT).glob(pref+'*.ipynb'))
        if found:names.insert(-2,found[0].name)
    for name in names:
        if not (project/GO2_ROOT/name).is_file():continue
        setup+="for cell in nbformat.read(project/"+repr(GO2_ROOT)+"/"+repr(name)+",4).cells:\n if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,"+repr(name)+",'exec'),globals())\n"
    setup+="go2_import_exports(Path("+repr(str(export_path))+"),globals())\ntorch.set_num_threads(2)\n"
    if export_path.name=='benchmark17_arena.ipynb':
        setup+="def go2_bench_sources(project):return {**go2_source_identity(project),'arena_draft':go2_code_sha(Path("+repr(str(export_path))+"))}\n"
    child=setup+helpers+"\n"+f"saved,_=go2_load_checkpoint(project/'checkpoints'/{prefix+'-resume'!r})\nmodel=Go2Network(Go2NetConfig(**saved['config']['network']));model.load_state_dict(saved['model']);model.eval()\nmetadata={metadata!r}\nconfig={cfg!r}\nplan={dict(plan,updated_at_utc='changed-presentation-only')!r}\nassert saved['search']['pending'] and any(j['status']=='PENDING' for j in saved['search']['reading_jobs'])\nresumed=go2_bench_arena(project,{prefix+'-resume'!r},config,plan,model,metadata,_Go2BenchPassFixture,_search_evaluator=_go2_bench_fixture_evaluator)\ngo2_atomic_json(Path({str(result_path)!r}),resumed)\n"
    nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(child)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}}),path)
    proc=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=180',str(path)],cwd=project,capture_output=True,text=True,timeout=240)
    if proc.returncode:raise RuntimeError(proc.stdout+'\n'+proc.stderr)
    resumed=json.loads(result_path.read_text())
    def decisions(report):
        return [{'outcome':g['outcome'],'state':g['state'],'trace':[{'kind':t['kind'],'color':t['color'],'action':t['action'],
            'visits':t.get('search_simulations'),'reading_nodes':t.get('reading_nodes'),'support':t.get('policy_support_simulations')} for t in g['trace']]} for g in report['games']]
    checks['fresh_kernel_exact_decisions']=decisions(full)==decisions(resumed)
    checks['presentation_timestamp_not_identity']=resumed['status']=='COMPLETE'
    cut=arena(project,prefix+'-cutoff',dict(cfg,max_plies=1),plan,model,metadata,factory,_search_evaluator=evaluator)
    checks['cutoff_keeps_UNKNOWN']=cut['summary']['UNKNOWN']==2 and cut['summary']['completed']==0
    timed=arena(project,prefix+'-clock',dict(cfg,game_time_seconds=1e-9),plan,model,metadata,factory,_search_evaluator=evaluator)
    checks['game_timeout_keeps_UNKNOWN']=timed['summary']['UNKNOWN']==2 and timed['summary']['completed']==0
    checks['paired_openings_replayed']=all(g['state']['actions'][:4]==[0,3,1,4] for g in resumed['games'])
    checks['draws_not_counted_as_wins']=_go2_bench_summary([dict(full['games'][0],outcome='DRAW')])['WIN']==0
    checks['UNKNOWN_denominator_explicit']=_go2_bench_summary([dict(full['games'][0],outcome='UNKNOWN'),dict(full['games'][1],outcome='WIN')])['win_rate_all_attempts_lower']==.5
    try:arena(project,prefix+'-resume',dict(cfg,batch_size=4),plan,model,metadata,factory,_search_evaluator=evaluator);checks['changed_config_refused']=False
    except RuntimeError:checks['changed_config_refused']=True
    stop_calls=[]
    def stop_callback(report):
        stop_calls.append(report['global_event']);raise KeyboardInterrupt('disposable user STOP fixture')
    try:arena(project,prefix+'-stop',cfg,plan,model,metadata,factory,on_progress=stop_callback,_search_evaluator=evaluator)
    except KeyboardInterrupt:pass
    stopped,_=go2_load_checkpoint(project/'checkpoints'/(prefix+'-stop'))
    checks['STOP_checkpoint_preserved_without_recursive_callback']=len(stop_calls)==1 and stopped is not None and stopped['self_play'] is not None
    # Real GNU Go transport plus a neural search. A short cutoff is not a strength result.
    real_namespace=globals().copy();go2_import_exports(export_path,real_namespace)
    if 'go2_bench_opponent_factory' not in real_namespace:
        go2_import_exports(project/'reports/ahr_v2/fixtures/benchmark17_opponents.ipynb',real_namespace)
    real_model=Go2Network(Go2NetConfig(board_size=9,width=16,blocks=1)).eval()
    real_metadata={'run_id':'ahr-v2-disposable-gnu-fixture','checkpoint_file':'disposable','checkpoint_sha256':go2_tensor_digest(real_model.state_dict()),
        'optimizer_step':0,'is_smoke':True,'model_identity':go2_model_identity(real_model)}
    real_cfg=dict(cfg,board_size=9,komi=7.5,max_plies=2,max_simulations=2,batch_size=2,ahr_move_seconds=.05)
    real_plan=[{'game_index':0,'profile':'gnugo-l10','opening_actions':[],'ahr_color':1,'seed':1701}]
    real_factory=lambda profile,seed,size,komi:real_namespace['go2_bench_opponent_factory'](project,profile,seed,size,komi,gtp_timeout_seconds=30,
        config_directory=project/'reports/ahr_v2/fixtures'/prefix)
    real_gnu=real_namespace['go2_bench_arena'](project,prefix+'-real-gnu',real_cfg,real_plan,real_model,real_metadata,real_factory)
    checks['real_GNU_neural_cutoff_keeps_UNKNOWN']=real_gnu['summary']['UNKNOWN']==1 and real_gnu['summary']['completed']==0 and real_gnu['games'][0]['plies']==2
    checks['real_GNU_trace_has_genmove']=any(t['kind']=='genmove' for t in real_gnu['games'][0]['trace'])
    report={'schema':'ahr-v2-benchmark-arena-validation-v1','status':'PASS' if all(checks.values()) else 'FAIL','checks':checks,
        'run_prefix':prefix,'fresh_fixture':str(path.relative_to(project)),'claim':'Short disposable correctness fixtures only. No production benchmark or training launched.'}
    go2_atomic_json(project/'reports/ahr_v2/benchmark17_arena_validation.json',report)
    if not all(checks.values()):raise AssertionError(checks)
    return report


## Detached execution and progress

In [ ]:
def go2_bench_validate(project,force=False):
    project=go2_project(project);sources=go2_bench_sources(project)
    path=project/'reports/ahr_v2/benchmark17_validation.json'
    assets=go2_bench_katago_assets(project,allow_download=False)
    pin=go2_digest({p:go2_bench_opponent_pin(project,p) for p in GO2_BENCH_PROFILES})
    if not force and path.is_file():
        receipt=json.loads(path.read_text())
        if receipt.get('status')=='PASS' and receipt.get('sources')==sources and receipt.get('opponent_pin_sha256')==pin:
            return receipt
    results={}
    for name,check in [('opponent_transport',go2_bench_validate_opponents),
                       ('paired_openings_and_speed_recovery',go2_bench_protocol_validation),
                       ('arena_recovery',go2_bench_arena_validation),
                       ('gpu_timing',go2_bench_gpu_validation),
                       ('detached_worker',go2_bench_validate_detached)]:
        print('Short validation:',name,flush=True)
        result=check(project)
        if not isinstance(result,dict) or (result.get('status') not in ['PASS','PASSED'] and result.get('all_passed') is not True):
            raise RuntimeError('Validation did not pass: '+name)
        results[name]=result
    receipt={'status':'PASS','sources':sources,'opponent_pin_sha256':pin,
             'checks':results,'validated_at_utc':go2_now(),
             'scope':'Disposable legal-history transport and exact clockless checkpoint recovery fixtures; no full strength benchmark or training.'}
    go2_atomic_json(path,receipt)
    return receipt

def go2_bench_status(project,runid):
    root=Path(project)/'reports/ahr_v2/benchmarks'/runid
    result={'run_id':runid,'status':'READY','worker':None,'arena':None,'speed':None,'plan':None}
    for key,filename in [('worker','worker.json'),('arena','report.json'),('speed','speed.json'),('plan','plan.json')]:
        path=root/filename
        if path.is_file():result[key]=json.loads(path.read_text())
    if result['worker']:result['status']=result['worker']['status']
    launch=root/'launch.json'
    if launch.is_file():
        session=json.loads(launch.read_text())['session']
        alive=subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0
        if result['worker'] is None:result['status']='STARTING' if alive else 'INTERRUPTED'
        elif result['status']=='RUNNING' and not alive:result['status']='INTERRUPTED'
    return result

def go2_bench_worker(project,config_path):
    project=go2_project(project);config=json.loads(Path(config_path).read_text());runid=config['run_id']
    root=project/'reports/ahr_v2/benchmarks'/runid;stop=root/'STOP_REQUESTED'
    try:
        if config['source_identity']!=go2_bench_sources(project):raise RuntimeError('Source exports changed; exact resume refused.')
        if config['opponent_pins']!={p['id']:go2_bench_opponent_pin(project,p['id']) for p in config['profiles']}:
            raise RuntimeError('Opponent assets changed; exact resume refused.')
        torch.set_num_threads(int(config.get('torch_threads',4)))
        model,metadata=go2_bench_pinned_model(project,config)
        if config['champion_pin']!={key:metadata[key] for key in config['champion_pin']}:
            raise RuntimeError('Frozen champion checkpoint changed.')
    except BaseException as error:
        go2_atomic_json(root/'worker.json',{'run_id':runid,'status':'ERROR','stage':'preflight','updated_at_utc':go2_now(),
            'configuration_sha256':go2_digest(config),'detail':{'error':repr(error)},'resume':'Restore the frozen sources/assets/checkpoint and Run All with the original settings.'})
        raise
    def note(stage,status='RUNNING',detail=None):
        record={'run_id':runid,'status':status,'stage':stage,'updated_at_utc':go2_now(),
                'configuration_sha256':go2_digest(config),'detail':detail,
                'resume':'Run All in notebook 17 with the same settings and run ID.'}
        go2_atomic_json(root/'worker.json',record)
        return record
    def progress(stage):
        def callback(report):
            note(stage,detail={k:report[k] for k in ['status','next_game_index','active_ply','completed_probes','planned_probes'] if k in report})
            if stop.is_file():raise KeyboardInterrupt('Graceful stop requested; the saved benchmark can resume.')
        return callback
    with go2_checkpoint_lock(project/'checkpoints'/(runid+'-control')):
        stage='opening_plan'
        try:
            if stop.is_file():raise KeyboardInterrupt('A stop request is pending.')
            note(stage);plan=go2_bench_plan(project,config,model,metadata)
            if plan['status']!='COMPLETE':raise RuntimeError('Opening generation did not finish.')
            stage='speed';note(stage)
            speed=go2_bench_speed(project,runid,config,model,metadata,plan,on_progress=progress(stage))
            if speed['status']!='COMPLETE' or speed['completed_probes']!=speed['planned_probes'] or len(speed['results'])!=speed['planned_probes'] or speed['planned_probes']<=0:
                raise RuntimeError('Speed stage is incomplete; refusing a completion claim.')
            errors=[row for row in speed['results'] if row['status']!='MEASURED']
            if errors:raise RuntimeError('Speed probes failed; inspect speed.json before interpreting results.')
            if stop.is_file():raise KeyboardInterrupt('Graceful stop requested.')
            stage='strength';note(stage)
            def factory(profile,seed,board_size,komi):
                return go2_bench_opponent_factory(project,profile,seed,board_size,komi,
                    gtp_timeout_seconds=config['gtp_timeout_seconds'],config_directory=root/'opponent_configs')
            arena=go2_bench_arena(project,runid,config,plan,model,metadata,factory,on_progress=progress(stage))
            if arena['status']!='COMPLETE' or len(arena['games'])!=len(plan['games']) or arena['next_game_index']!=len(plan['games']):raise RuntimeError('Arena returned before all attempts were recorded.')
            stage='finished'
            return note(stage,'COMPLETE',{'summary':arena['summary'],'speed_summary':speed['summary'],
                    'interpretation':'Observed development results; paired openings are correlated. CPU opponents and timed GPU AHR use different compute budgets.'})
        except KeyboardInterrupt as error:
            return note(stage,'STOPPED',{'reason':str(error)})
        except BaseException as error:
            note(stage,'ERROR',{'error':repr(error)})
            raise

def go2_launch_benchmark(project,config,session='ahr-v2-benchmark17',resume_stopped=False):
    project=go2_project(project)
    if not shutil.which('tmux'):raise FileNotFoundError('tmux is required for detached execution.')
    frozen=go2_bench_frozen_config(project,config);runid=config['run_id']
    receipt=go2_bench_validate(project)
    if receipt['sources']!=frozen['source_identity']:raise RuntimeError('Validation is stale.')
    root=project/'reports/ahr_v2/benchmarks'/runid;root.mkdir(parents=True,exist_ok=True)
    config_path=project/'config/ahr_v2_benchmarks'/(runid+'.json')
    with go2_checkpoint_lock(project/'checkpoints'/(runid+'-launch')):
        if config_path.is_file():
            if json.loads(config_path.read_text())!=frozen:
                raise RuntimeError('Existing run configuration differs. Restore its settings or use a new run ID.')
        else:
            if any((project/'checkpoints'/(runid+suffix)/'manifest.json').exists() for suffix in ['', '-plan','-speed']):
                raise RuntimeError('Checkpoint exists without its frozen launch configuration; refusing a silent restart.')
            go2_atomic_json(config_path,frozen)
        running=subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0
        launch_path=root/'launch.json'
        if running:
            if not launch_path.is_file() or json.loads(launch_path.read_text()).get('run_id')!=runid:
                raise RuntimeError('Requested tmux session belongs to another launch.')
            return {'status':'RUNNING','run_id':runid,'session':session,'attach_command':'tmux attach -t '+session}
        current=go2_bench_status(project,runid)
        if current['status']=='COMPLETE':return {'status':'COMPLETE','run_id':runid,'report':str(root/'report.json')}
        stop=root/'STOP_REQUESTED'
        if stop.exists():
            if not resume_stopped:raise RuntimeError('Run was stopped. Set RESUME_STOPPED_RUN=True to resume the same checkpoints.')
            stop.unlink()
        worker=root/'detached_worker.ipynb';log=root/'tmux.log'
        lines=['from pathlib import Path','import nbformat','PROJECT=Path('+repr(str(project))+')',
               'for name in ["00_contracts_and_recovery.ipynb","15_teacher_evaluation_and_launch.ipynb"]:',
               '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
               '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]): exec(compile(cell.source,name+":ahr-export","exec"),globals())',
               'globals().update(go2_teacher_arena_namespace(PROJECT))',
               'for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/'+repr(GO2_BENCH_BOOK)+',4).cells:',
               '    if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]): exec(compile(cell.source,"17:ahr-export","exec"),globals())',
               'result=go2_bench_worker(PROJECT,Path('+repr(str(config_path))+'))',
               'print("Benchmark worker returned:",result["status"])']
        book=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}},
            cells=[nbformat.v4.new_markdown_cell('# Notebook 17 detached benchmark worker'),nbformat.v4.new_code_cell(chr(10).join(lines))])
        go2_atomic_bytes(worker,nbformat.writes(book).encode())
        launch={'status':'LAUNCHING','run_id':runid,'session':session,'launched_at_utc':go2_now(),
                'configuration_sha256':go2_digest(frozen),'worker':str(worker.relative_to(project)),
                'log':str(log.relative_to(project)),'attach_command':'tmux attach -t '+session,'detach_keys':'Ctrl-b, then d',
                'resume_cell':'17: Run All with identical run ID and settings; completed stages remain saved.'}
        go2_atomic_json(launch_path,launch)
        args=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
              '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(worker)]
        command=' '.join(shlex.quote(arg) for arg in args)+' >> '+shlex.quote(str(log))+' 2>&1'
        subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True,capture_output=True,text=True)
        launch['status']='LAUNCHED';go2_atomic_json(launch_path,launch)
        return launch

def go2_bench_request_stop(project,runid):
    path=Path(project)/'reports/ahr_v2/benchmarks'/runid/'STOP_REQUESTED'
    go2_atomic_bytes(path,b'Graceful stop requested from notebook 17.\n')
    return 'Stop requested. The worker will checkpoint and stop at its next safe callback.'

def go2_bench_display(project,config):
    from IPython.display import display,Markdown
    import pandas as pd
    state=go2_bench_status(project,config['run_id'])
    display(Markdown('**Benchmark status: '+state['status']+'** · '+config['run_id']))
    if state['worker']:
        display(Markdown('Stage: **'+state['worker']['stage']+'** · Updated '+go2_bench_dublin(state['worker']['updated_at_utc'])))
        if state['worker'].get('status') in ['ERROR','STOPPED']:display(Markdown(str(state['worker'].get('detail'))))
    rows=[{'Opponent':p['id'],'Planned games':p['games'],'AHR Black':p['games']//2,'AHR White':p['games']//2} for p in config['profiles']]
    display(pd.DataFrame(rows).set_index('Opponent'))
    if state['plan']:display(Markdown('Opening plan: **'+str(state['plan']['coverage']['fresh_opening_families'])+' fresh families**, '+str(state['plan']['coverage']['empty_board_pairs'])+' empty-board baseline pairs.'))
    if state['arena']:
        rows=[]
        for profile,summary in state['arena']['by_profile'].items():
            rows.append({'Opponent':profile,'Attempts':summary['attempts'],'AHR wins':summary['WIN'],'Losses':summary['LOSS'],
                         'Draws':summary['DRAW'],'UNKNOWN':summary['UNKNOWN'],
                         'Win rate, completed':None if summary['win_rate_completed'] is None else round(100*summary['win_rate_completed'],1)})
        display(pd.DataFrame(rows).set_index('Opponent'))
        display(Markdown('Completed-game rates exclude UNKNOWN; every cutoff remains visible. Colour-paired games share openings. These observations do not establish a rank or Elo.'))
    else:
        display(Markdown('**No full benchmark results yet.** Run All starts this suite in tmux. Saved validation outputs come from disposable smoke fixtures.'))
    if state['speed']:
        display(pd.DataFrame(state['speed']['summary']))
    display(Markdown('AHR: GPU `'+config['device']+'`, **'+str(config['ahr_move_seconds'])+' s soft target**, up to **'+str(config['max_simulations'])+' simulations**. KataGo: single-thread CPU EIGEN, fixed visit caps. Budgets and hardware differ.'))
    return state

def go2_bench_gpu_validation(project):
    if not torch.cuda.is_available():raise RuntimeError('GPU timing validation requires the configured GPU.')
    config=go2_bench_default_config();config.update(run_id='ahr-v2-benchmark-validation-gpu-'+uuid.uuid4().hex[:8],
        speed_positions=3,speed_devices=['cuda:0'],speed_move_seconds=[0.03],speed_max_simulations=8,batch_size=2)
    model,metadata=go2_bench_pinned_model(project,config,'cpu')
    plan={'identity':{'purpose':'disposable_gpu_timing_fixture'},'games':[],'coverage':{}}
    partial=go2_bench_speed(project,config['run_id'],config,model,metadata,plan,max_probes=1)
    resumed=go2_bench_speed(project,config['run_id'],config,model,metadata,plan)
    retained=partial['results']==resumed['results'][:len(partial['results'])]
    measured=all(row['status']=='MEASURED' and row['completed_simulations']>0 for row in resumed['results'])
    if not retained or not measured or resumed['completed_probes']!=3:raise AssertionError('GPU timing recovery fixture failed.')
    return {'status':'PASS','run_id':config['run_id'],'completed_measurements_retained':True,
        'cuda_synchronization_exercised':True,'probe_count':3,
        'max_seconds':max(row['elapsed_seconds'] for row in resumed['results']),
        'scope':'Short disposable GPU timing measurements, not the main 1/2-second speed suite or strength matches.'}

def go2_bench_plots(status):
    from IPython.display import display,Markdown
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
    arena=status.get('arena');speed=status.get('speed')
    if arena and arena['games']:
        profiles=list(arena['by_profile']);left=np.zeros(len(profiles));fig,ax=plt.subplots(figsize=(9,3.8))
        for label,color,hatch in [('WIN','#356D99',''),('LOSS','#D39239','//'),('DRAW','#AEAFB1','..'),('UNKNOWN','#EBEBEB','xx')]:
            values=[arena['by_profile'][profile][label] for profile in profiles]
            ax.barh(profiles,values,left=left,label=label,color=color,hatch=hatch,edgecolor='#555555',linewidth=.6);left+=values
        ax.set_xlabel('Recorded game attempts');ax.set_xlim(left=0);scope='validation fixtures' if arena.get('purpose')=='validation_fixture' else 'development benchmark'
        ax.set_title('AHR outcomes by opponent — '+scope,pad=44)
        from matplotlib.ticker import MaxNLocator
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend(ncol=4,loc='lower left',bbox_to_anchor=(0,1.01),frameon=False)
        fig.tight_layout();display(fig);plt.close(fig)
        display(Markdown('Source: saved `report.json`. UNKNOWN is shown separately; paired colours share an opening and do not provide independent evidence of rank.'))
    if speed and any(row['status']=='MEASURED' for row in speed['results']):
        rows=[row for row in speed['summary'] if row['measured']]
        labels=[row['device']+' / '+str(row['move_seconds'])+' s target' for row in rows]
        ypos=np.arange(len(rows));fig,(ax,vis)=plt.subplots(1,2,figsize=(11,3.8))
        for values,marker,label,color in [('median_seconds','o','Median','#356D99'),('p95_seconds','s','p95','#D39239'),('max_seconds','x','Maximum','#555555')]:
            ax.scatter([row[values] for row in rows],ypos,marker=marker,label=label,color=color)
        ax.set_yticks(ypos,labels);ax.set_xlabel('End-to-end move time (seconds)');ax.set_xlim(left=0);ax.set_title('Measured move times');ax.legend(frameon=False)
        vis.barh(labels,[row['median_visits'] for row in rows],color='#356D99');vis.set_xlabel('Median completed simulations');vis.set_title('Search work achieved');vis.set_xlim(left=0)
        fig.tight_layout();display(fig);plt.close(fig)
        display(Markdown('Source: saved `speed.json`; warmed process, compatible validation positions. p95 describes the measured samples. Soft budgets may overrun; future timing is not guaranteed.'))


## Validate setup

A short cached validation checks legal opponent transport, fresh-process replay, checkpoint creation and exact clockless recovery. It does not run the 100-game suite.

In [ ]:
VALIDATION=go2_bench_validate(PROJECT)
display(Markdown('**Setup and recovery validation: '+VALIDATION['status']+'** · '+go2_bench_dublin(VALIDATION['validated_at_utc'])))


## Run or resume

**Run All** launches session `ahr-v2-benchmark17`, then returns to the notebook. Rerunning with the same settings shows an active session, resumes an interrupted run, or displays completed results. A different model, source, asset or configuration is refused for an existing run ID.

Attach from a terminal: `tmux attach -t ahr-v2-benchmark17`. Detach with **Ctrl-b**, then **d**. Progress and logs stay under the run report directory. Disconnecting your browser does not stop tmux.

Checkpoints are saved at each game/stage boundary and within 120 seconds or 250 search events; manifests retain at least the latest two valid states and the best. Incomplete search/reader state and opponent command history resume from checkpoints. Wall-clock scheduling may change future visit counts; clockless recovery fixtures verify the underlying state and action choices.

Use the **Request stop** button for a checkpointed stop. To resume a deliberately stopped run, set `RESUME_STOPPED_RUN=True`, keep the original settings and click Run All. Files remain on the VM persistent boot disk; off-instance backup is deferred as requested.

In [ ]:
if EXECUTION_MODE=='benchmark':
    LAUNCH=go2_launch_benchmark(PROJECT,BENCHMARK_CONFIG,TMUX_SESSION,RESUME_STOPPED_RUN)
    display(Markdown('**'+LAUNCH['status']+'** · '+LAUNCH['run_id']))
    if LAUNCH.get('attach_command'):print(LAUNCH['attach_command'])
else:
    display(Markdown('**Validation mode: full benchmark was not launched.** On your normal Run All the default mode is benchmark.'))
BENCHMARK_STATUS=go2_bench_display(PROJECT,BENCHMARK_CONFIG)

go2_bench_plots(BENCHMARK_STATUS)


In [ ]:
import ipywidgets as widgets
refresh=widgets.Button(description='Refresh results',icon='refresh')
stop_button=widgets.Button(description='Request stop',icon='stop',button_style='warning')
progress_output=widgets.Output()
def refresh_benchmark(_):
    with progress_output:
        progress_output.clear_output(wait=True)
        status=go2_bench_display(PROJECT,BENCHMARK_CONFIG)
        go2_bench_plots(status)
def stop_benchmark(_):
    with progress_output:
        print(go2_bench_request_stop(PROJECT,BENCHMARK_CONFIG['run_id']))
refresh.on_click(refresh_benchmark);stop_button.on_click(stop_benchmark)
display(widgets.HBox([refresh,stop_button]),progress_output)
